# 🧬 Human Cytokine Dictionary — Immunology Practical (v3 · immune genes)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/garridolab/cytokines-practical/blob/main/CytokineDictionary_Practical_v3.ipynb)

**Based on:** Oesinghaus *et al.* (2025), *A single-cell cytokine dictionary of human peripheral blood*, bioRxiv
([doi:10.64898/2025.12.12.693897](https://doi.org/10.64898/2025.12.12.693897))

In the paper, ~9.7 million human blood cells (PBMCs) from **12 donors** were stimulated with **90 cytokines** (or with a PBS control)
and read out by single-cell RNA-seq. Here you will work with a **pseudobulk** version of the data: for every
*donor × cytokine × cell type*, the counts of all cells were **summed** into one bulk-like expression profile.
We use **four cell types**: **CD4 T cells, CD8 T cells, NK (CD56low) cells and CD14 monocytes**.
(The file we use contains **87 of the cytokines** plus the PBS control, for the 12 donors.)

> 🧬 **Immune-genes version.** Parts 1–3 are identical to the standard practical. In **Part 4** the differential-expression analysis is restricted to **~500 expressed genes related to the immune system** (section 4.0), instead of all ~12,700 expressed genes. The text, the questions and their answers have been adapted to this smaller gene set.

### The plan (2 h 30 min)

| Part | Topic | Format | Time |
|:--:|---|---|:--:|
| **1** | **Introduction**: Python, Jupyter & Colab, RNA-seq, single-cell, pseudobulk, the dataset, AnnData | slides + 5 min of setup here | 25 min |
| **2** | **Manipulating and normalizing the data** | this notebook | 40 min |
| ☕ | *Break* | | 10 min |
| **3** | **Principal Component Analysis (PCA)**: one analysis to rule them all | slides + notebook | 35 min |
| **4** | **Differential expression**: cytokine vs. control | slides + notebook | 35 min |
| | Wrap-up | | 5 min |

### How to read this notebook

| Symbol | Meaning |
|:--:|---|
| ⏱ | Time budget of the section |
| 🔧 | **Your turn / CHANGE ME**: a task for you. Code cells that start with `# 🔧 CHANGE ME` contain parameters you can modify: change them, re-run and see what happens |
| ❓ | **Question**: think about it *before* opening the answer |
| 💡 | Tip / something to remember |
| ⚠️ | A common trap |
| 🎁 | Bonus: only if you have time (or at home) |

> 📑 **Every exercise has its own heading** (`Exercise 2.2a`, `Exercise 3.3`…), so you can jump to it from the **table of contents** (in Colab: the ☰ icon on the left; in Jupyter: *View → Table of Contents*).

> **Running the notebook:** execute the cells **from top to bottom** with **Shift + Enter**.
> If you get lost, there is a **catch-up cell** at the start of Part 3: it rebuilds the data from Part 2 in one go.

---
# Part 1 · Introduction

The ideas (Python, Jupyter, Colab, RNA-seq, single-cell, pseudobulk, the dataset, AnnData) are explained in the **slides**.
Here you only need to **get ready** — it takes about 5 minutes. ⏱ *5 min*

## 1.1 · Jupyter in 60 seconds

* A notebook is a list of **cells**. Some contain **text** (Markdown, like this one) and some contain **code** (Python).
* To **run** a code cell: click on it and press **Shift + Enter** (or the ▶ button). Its output appears right below.
* Cells share their **memory**: a variable created in one cell exists in all the following ones. Run cells **in order**.
* Something went wrong? **Runtime → Restart session**, then run the cells again from the top.

Try it:

In [ ]:
# This is a code cell: click on it and press Shift+Enter.
# It creates a variable called `message`, which now lives in the notebook's memory.
# 🔧 CHANGE ME: write your own name
message = "Hello, immunologists!"

In [ ]:
print(message)          # this cell uses the variable created in the cell above
6 * 7                   # the value of the LAST line of a cell is displayed automatically

#### 🔧 Exercise 1.1 · Your turn

> Change `message` to your own name, run the first cell and then the second one.
> Then do **Runtime → Restart session** and run *only the second cell*: you get a `NameError` — **order matters**.

## 1.2 · Install and import the packages

Colab already includes most of what we need. The cell below installs only what is missing (`anndata`, …).

In [ ]:
# ── Install any missing dependencies ─────────────────────────────────────────
import importlib.util, shutil, subprocess, sys

REQUIRED = {                       # import name -> PyPI package name
    "numpy": "numpy", "pandas": "pandas", "scipy": "scipy", "matplotlib": "matplotlib",
    "sklearn": "scikit-learn", "statsmodels": "statsmodels", "anndata": "anndata",
    "gdown": "gdown", "adjustText": "adjustText", "pydeseq2": "pydeseq2",
}
missing = [pkg for mod, pkg in REQUIRED.items() if importlib.util.find_spec(mod) is None]

if missing:
    print(f"📦 Installing: {', '.join(missing)} …")
    if shutil.which("uv"):
        cmd = ["uv", "pip", "install", "--python", sys.executable, *missing]
    else:
        cmd = [sys.executable, "-m", "pip", "install", "-q", *missing]
    subprocess.check_call(cmd)
    importlib.invalidate_caches()
    print("✅ Done")
else:
    print("✅ Everything is already installed")

In [ ]:
#@title Imports, colours and settings (run once, no need to read) { display-mode: "form" }
import os, warnings
import numpy as np
import pandas as pd
import anndata as ad
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from matplotlib.colors import LinearSegmentedColormap
from scipy import stats
from statsmodels.stats.multitest import multipletests
from sklearn.decomposition import PCA
from IPython.display import display

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)
rng = np.random.default_rng(42)                # one random generator for the whole notebook

# ── Plot style ───────────────────────────────────────────────────────────────
plt.rcParams.update({
    "figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
    "axes.titleweight": "bold", "axes.titlesize": 10.5, "axes.labelsize": 9.5,
    "legend.frameon": False, "legend.fontsize": 8,
})

# ── The four cell types and the 12 donors ────────────────────────────────────
# Colours come from a colour-blind-checked categorical palette; every cell type also has its own
# marker shape, so identity never relies on colour alone.
CELL_TYPES = ["CD4_T_cell", "CD8_T_cell", "NK_CD56low", "CD14_Mono"]
CT_LABEL   = {"CD4_T_cell": "CD4 T cell", "CD8_T_cell": "CD8 T cell", "NK_CD56low": "NK CD56low", "CD14_Mono": "CD14 monocyte"}
CT_SHORT   = {"CD4_T_cell": "CD4 T", "CD8_T_cell": "CD8 T", "NK_CD56low": "NK", "CD14_Mono": "Mono"}
CT_COLORS  = {"CD4_T_cell": "#2a78d6", "CD8_T_cell": "#008300", "NK_CD56low": "#eda100", "CD14_Mono": "#e87ba4"}
CT_MARKERS = {"CD4_T_cell": "o", "CD8_T_cell": "s", "NK_CD56low": "^", "CD14_Mono": "D"}
DONORS     = [f"Donor{i}" for i in range(1, 13)]

# ── Colours for "direction": blue (down) ↔ grey ↔ red (up) ───────────────────
UP_COLOR, DOWN_COLOR, NS_COLOR, PBS_COLOR = "#e34948", "#2a78d6", "#c3c2b7", "#8a8984"
DIVERGING  = LinearSegmentedColormap.from_list("div", [DOWN_COLOR, "#f0efec", UP_COLOR])
SEQUENTIAL = LinearSegmentedColormap.from_list("seq", ["#f4f8fd", "#86b6ef", "#2a78d6", "#0d366b"])

print("✅ Imports OK")

## 1.3 · Download and load the data

The data are one `.h5ad` file (≈ 500 MB, an **AnnData** object saved to disk): the pseudobulk counts of 4 cell types,
restricted to 18,120 protein-coding genes. On Colab the download usually takes a minute or two.

In [ ]:
# Google Drive file with the pseudobulk data (shared as "anyone with the link")
GDRIVE_FILE_ID = "1thFoQnNVVqgKvspg5b5k2fBgjk4bEqAn"
# /content exists on Colab; locally the file is saved next to the notebook
DATA_PATH = "/content/pseudobulks_4ct_proteincoding.h5ad" if os.path.isdir("/content") else "pseudobulks_4ct_proteincoding.h5ad"

if not os.path.exists(DATA_PATH):
    import gdown
    print("⬇️  Downloading from Google Drive …")
    os.makedirs(os.path.dirname(os.path.abspath(DATA_PATH)), exist_ok=True)
    out = gdown.download(f"https://drive.google.com/uc?id={GDRIVE_FILE_ID}", DATA_PATH, quiet=False)
    if out is None or not os.path.exists(DATA_PATH):
        raise RuntimeError("Download failed: try again in a minute (Drive quota), or ask for the file.")
print(f"✅ File ready: {DATA_PATH}  ({os.path.getsize(DATA_PATH) / 1e9:.2f} GB)")

adata_raw = ad.read_h5ad(DATA_PATH)        # read the file into memory: an AnnData object
print("✅ Loaded into memory")

That's all for Part 1 on the notebook side. 👉 **Back to the slides** for the introduction; when we reach the *AnnData* slide, come back here.

---
# Part 2 · Manipulating and normalizing the data
⏱ *40 min* — **Goal:** understand what is inside the data, and turn raw counts into numbers we can compare.

| Step | Question | Time |
|:--:|---|:--:|
| **2.1** | Prepare the object | 3 min |
| **2.2** | What does the data look like? From **1 number** to the **whole matrix** | 10 min |
| **2.3** | What is a **count**? Why do libraries have different sizes? | 7 min |
| **2.4** | **Normalization**: what, why, how | 10 min |
| **2.5** | Why do we **filter** lowly expressed genes? | 5 min |
| **2.6** | **Heatmaps** before and after normalizing | 5 min |

## 2.1 · Prepare the object

The file we loaded contains a few leftovers of the original single-cell object. This cell tidies it up
(we explain *why* in the 🎁 bonus *"The PBS puzzle"* at the end — for now just run it):

1. the **PBS control** was measured in 6 wells, and the file stores each well *plus* their total → we keep the **total** only;
2. we keep the **sample metadata** we need and give every sample a readable, **unique name**: `donor|cell type|cytokine`;
3. we convert the matrix to a plain (dense) table of numbers;
4. we add two **gene-level** columns of our own: total counts and in how many samples the gene is detected.

In [ ]:
# 1) PBS control: keep the pre-computed total row, drop the 6 individual wells that were summed into it
is_pbs_well = ((adata_raw.obs["cytokine"] == "PBS") & adata_raw.obs["bc1_well"].notna()).values
adata = adata_raw[~is_pbs_well].copy()
if "psbulk_props" in adata.layers:
    del adata.layers["psbulk_props"]                 # a layer we do not use

# 2) Sample metadata (obs): keep the useful columns, set the order of the categories, give unique names
obs = adata.obs[["donor", "cytokine", "treatment", "cell_type", "psbulk_n_cells"]].copy()
cytokines = ["PBS"] + sorted(c for c in obs["cytokine"].astype(str).unique() if c != "PBS")
obs["donor"]     = pd.Categorical(obs["donor"].astype(str), categories=DONORS)
obs["cytokine"]  = pd.Categorical(obs["cytokine"].astype(str), categories=cytokines)
obs["cell_type"] = pd.Categorical(obs["cell_type"].astype(str), categories=CELL_TYPES)
obs["treatment"] = pd.Categorical(np.where(obs["cytokine"] == "PBS", "control", "treated"), categories=["control", "treated"])
obs.index = (obs["donor"].astype(str) + "|" + obs["cell_type"].astype(str) + "|" + obs["cytokine"].astype(str)).values
assert obs.index.is_unique
adata.obs = obs

# 3) The matrix X: raw counts as a dense table of numbers (80 % of the entries are non-zero)
X = adata.X.toarray() if hasattr(adata.X, "toarray") else np.asarray(adata.X)
adata.X = X.astype(np.float32)

# 4) Gene metadata (var): start from scratch and add two gene-level columns computed from the counts
adata.var = pd.DataFrame(index=adata.var_names)
adata.var["total_counts"]       = adata.X.sum(axis=0)          # counts of the gene, summed over all samples
adata.var["n_samples_detected"] = (adata.X > 0).sum(axis=0)    # in how many samples the gene has at least 1 count

print("✅ adata is ready")

## 2.2 · One number at a time: from 1 × 1 to the whole matrix

An **AnnData** object (the slides explained it) keeps three tables **aligned**: `X` (the numbers), `obs` (one row per sample) and `var` (one row per gene).
We select pieces with `adata[ samples , genes ]` — **samples first, genes second** — using *names*.

### The smallest piece: 1 sample × 1 gene

In [ ]:
# 🔧 CHANGE ME: any sample ("Donor7|CD4_T_cell|IL-2") and any gene ("CD3E", "IL7R", ...)
sample = "Donor1|CD14_Mono|IFN-gamma"     # donor | cell type | cytokine
gene   = "STAT1"

one = adata[[sample], [gene]]             # adata[ list of samples , list of genes ]
print(one)

In [ ]:
print("X: the measurement\n")
display(one.to_df())                      # a 1 × 1 table: this sample, this gene

print("obs: metadata of the SAMPLE (one row per sample)\n")
display(one.obs)

print("var: metadata of the GENE (one row per gene)\n")
display(one.var)

**What are we looking at?**

* `X` is **one number**: the **count** of the gene `STAT1` in this sample: how many mRNA molecules (UMIs) were detected in *all* CD14 monocytes of Donor 1 stimulated with IFN-γ, added together.
* `obs` describes the **sample**: the donor, the cytokine, whether it is a control or treated, the cell type and **how many cells were summed** (`psbulk_n_cells`).
* `var` describes the **gene**: here, its total counts over all samples and in how many samples it was detected.

### One sample, five genes (1 × 5)

The same sample, now with five genes: three **cell-type markers** (`LYZ`: monocytes, `CD3E`: T cells, `NKG7`: NK cells),
an **interferon-response gene** (`STAT1`) and a **housekeeping gene** (`ACTB`, expected to be always on).

In [ ]:
# 🔧 CHANGE ME: any five genes you like
genes5 = ["LYZ", "CD3E", "NKG7", "STAT1", "ACTB"]
five_genes = adata[[sample], genes5]
print(five_genes)                         # 1 sample × 5 genes: AnnData always writes  n_obs × n_vars  (samples × genes)

display(five_genes.to_df())               # one row (the sample), five columns (the genes)
display(five_genes.var)                   # five rows: one per gene

### Five samples, five genes (5 × 5)

Now five **samples** of Donor 1: the **control (PBS)** of each of the four cell types, plus the monocytes stimulated with IFN-γ.

In [ ]:
# 🔧 CHANGE ME: another donor ("Donor5") or another cytokine instead of IFN-gamma
samples5 = [f"Donor1|{ct}|PBS" for ct in CELL_TYPES] + ["Donor1|CD14_Mono|IFN-gamma"]
five_by_five = adata[samples5, genes5]
print(five_by_five)

print("\nX: raw counts (rows = samples, columns = genes)")
display(five_by_five.to_df())
print("obs: the five samples")
display(five_by_five.obs)

#### ❓ Exercise 2.2a · Questions
1. For `LYZ`, `CD3E` and `NKG7`: which sample has the highest value? Does it match what you know about these markers?
2. `ACTB` should be "always on", yet its counts differ a lot between rows. Look at `psbulk_n_cells`: do you see a link?
3. `STAT1` is induced by IFN-γ. Compare the two **monocyte** rows (PBS vs IFN-γ): is the raw `STAT1` count higher with IFN-γ? Is that what you expected?

<details>
<summary>💡 Answers</summary>

1. `LYZ` → monocytes; `CD3E` → T cells (the CD4 T sample has the highest value, CD8 T the second); `NKG7` → NK cells, then CD8 T cells (also cytotoxic).
2. Yes: more cells summed → more counts. Compare the two monocyte rows: 17,373 cells vs 2,556 cells → 260,422 vs 36,500 counts of `ACTB`.
3. **No — it is ~9× *lower* with IFN-γ** (73,853 vs 639,289), although IFN-γ induces `STAT1`. The PBS sample is the sum of 6 wells and has 7× more cells.
   **Raw counts mix biology with the amount of material**: that is the point of the next two sections.

</details>

### Selecting with a condition (boolean mask)

Names are fine for a few samples. To select *many*, build a **boolean mask** on `obs` — a True/False value for every sample — and use it as the first index.
Here: all monocyte samples that are control or IFN-γ-treated (12 donors × 2 = 24 samples), for two interferon-γ targets, `IDO1` and `CXCL10`:

In [ ]:
# 🔧 CHANGE ME: the cell type, the cytokines and the genes
mask = (adata.obs["cell_type"] == "CD14_Mono") & adata.obs["cytokine"].isin(["PBS", "IFN-gamma"])
sub  = adata[mask, ["IDO1", "CXCL10"]]
print(sub, "\n")

table = sub.to_df()
table["donor"]    = sub.obs["donor"].astype(str).values
table["cytokine"] = sub.obs["cytokine"].astype(str).values
display(table.pivot(index="donor", columns="cytokine", values=["IDO1", "CXCL10"]).loc[DONORS])    # 12 donors × (2 genes × 2 conditions)

Look carefully: for several donors the **PBS counts are as high as, or higher than, the IFN-γ counts** — yet IFN-γ is a strong inducer of both genes!
The PBS samples contain ~5× more cells (they are the sum of 6 wells), so they have more counts of *everything*. Raw counts cannot be compared between samples; we'll fix that in 2.3–2.4.

### Zooming out: the complete matrix

In [ ]:
print(adata)
print("\nShape of the complete matrix (samples × genes):", adata.shape)
print(f"→ {adata.n_obs:,} samples × {adata.n_vars:,} genes = {adata.n_obs * adata.n_vars:,} numbers")
print(f"→ the design: 4 cell types × 88 conditions (87 cytokines + PBS) × 12 donors = {4 * 88 * 12:,} samples")
print(f"→ {np.mean(adata.X == 0):.1%} of the entries are zeros")
print(f"→ {adata.X.nbytes / 1e6:.0f} MB in memory\n")

display(pd.crosstab(adata.obs["cell_type"], adata.obs["treatment"]))

The 5 × 5 table we printed before was just the **top-left corner** of this big table. Everything we do next is applied to the *whole* matrix at once.

#### 🔧 Exercise 2.2b · Your turn (2 min)

> 1. How many `NK_CD56low` samples were treated with `IL-2`? *(hint: build a mask with `&`, then take `.n_obs` of the subset)*
> 2. Get the raw counts of `CXCL9` in the 12 `CD14_Mono` samples treated with `IL-4`. Which donor has the highest value?

<details>
<summary>💡 Solutions</summary>

```python
m = (adata.obs["cell_type"] == "NK_CD56low") & (adata.obs["cytokine"] == "IL-2")
adata[m].n_obs                                    # 12: one sample per donor

m = (adata.obs["cell_type"] == "CD14_Mono") & (adata.obs["cytokine"] == "IL-4")
adata[m, ["CXCL9"]].to_df().sort_values("CXCL9", ascending=False)
```

</details>

In [ ]:
# 🔧 your code here

## 2.3 · What is a count? Library sizes

A **count** is the number of **UMIs** (*unique molecular identifiers*: one UMI = one original mRNA molecule, after removing PCR duplicates)
that were assigned to a gene. In our pseudobulk, that number is **summed over all the cells** of the sample.
So a count is a **non-negative integer** that depends on two things:

1. how much of the gene was **expressed** (the biology we want), and
2. how much **material** we have: how many cells, and how deeply they were sequenced (technical).

### The raw data at a glance

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11.5, 3.9))

# (left) all the counts of 150 random samples
vals = adata.X[rng.choice(adata.n_obs, 150, replace=False)].ravel()
axes[0].hist(np.log10(vals + 1), bins=80, color="#2a78d6")
axes[0].set_xlabel("log10(count + 1)"); axes[0].set_ylabel("number of (sample, gene) entries")
axes[0].set_title(f"Counts: {np.mean(vals == 0):.0%} zeros, and values up to {vals.max():,.0f}")

# (right) a few genes take a big share of all counts
gene_total = adata.var["total_counts"].values
order = np.argsort(gene_total)[::-1]
share = np.cumsum(gene_total[order]) / gene_total.sum() * 100
axes[1].plot(np.arange(1, len(share) + 1), share, color="#2a78d6", lw=2)
axes[1].set_xscale("log")
for k in (10, 100, 1000):
    axes[1].plot(k, share[k - 1], "o", color=UP_COLOR)
    axes[1].annotate(f"top {k:,} genes: {share[k - 1]:.0f}%", (k, share[k - 1]), xytext=(-8, 8), textcoords="offset points", ha="right", fontsize=8)
axes[1].set_xlabel("genes, sorted by total counts"); axes[1].set_ylabel("cumulative % of all counts")
axes[1].set_title("1,000 genes (of 18,120) hold half of all counts")
plt.tight_layout(); plt.show()

### Library size: how many counts does each sample have?

The **library size** of a sample is the sum of all its counts: $L_i=\sum_j \text{count}_{ij}$. Let's compare the four cell types.

In [ ]:
adata.obs["lib_size"]      = adata.X.sum(axis=1)                              # total counts of each sample
adata.obs["umis_per_cell"] = adata.obs["lib_size"] / adata.obs["psbulk_n_cells"]

obs     = adata.obs
is_ctrl = (obs["treatment"] == "control").values

fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.5))

# (left) library size of every sample, by cell type  (circles = treated samples; open rings = control/PBS)
ax = axes[0]
for k, ct in enumerate(CELL_TYPES):
    m = (obs["cell_type"] == ct).values
    v, vp = obs.loc[m & ~is_ctrl, "lib_size"].values, obs.loc[m & is_ctrl, "lib_size"].values
    ax.scatter(k - 0.1 + rng.uniform(-0.2, 0.2, len(v)), v, s=7, alpha=0.4, linewidths=0, color=CT_COLORS[ct], marker=CT_MARKERS[ct])
    ax.scatter(k + 0.33 + rng.uniform(-0.06, 0.06, len(vp)), vp, s=30, facecolors="none", edgecolors="k", linewidths=0.9)
    ax.hlines(np.median(v), k - 0.4, k + 0.2, color="k", lw=2)
    ax.text(k - 0.1, 1.5e8, f"median\n{np.median(v) / 1e6:.1f} M", ha="center", fontsize=8)
ax.set_yscale("log"); ax.set_ylim(5e4, 6e8)
ax.set_xticks(range(4)); ax.set_xticklabels([CT_LABEL[c].replace(" ", "\n", 1) for c in CELL_TYPES])
ax.set_ylabel("library size (total counts)"); ax.set_title("Library size by cell type  (○ = PBS controls)")

# (right) library size follows the number of cells that were summed
ax = axes[1]
for ct in CELL_TYPES:
    m = (obs["cell_type"] == ct).values & ~is_ctrl
    ax.scatter(obs.loc[m, "psbulk_n_cells"], obs.loc[m, "lib_size"], s=7, alpha=0.4, linewidths=0, color=CT_COLORS[ct], marker=CT_MARKERS[ct], label=CT_LABEL[ct])
ax.scatter(obs.loc[is_ctrl, "psbulk_n_cells"], obs.loc[is_ctrl, "lib_size"], s=30, facecolors="none", edgecolors="k", linewidths=0.9, label="PBS controls")
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel("number of cells in the pseudobulk"); ax.set_ylabel("library size (total counts)")
ax.set_title("Library size ≈ number of cells × UMIs per cell"); ax.legend(markerscale=1.8, loc="upper left")
plt.tight_layout(); plt.show()

summary = obs[~is_ctrl].groupby("cell_type", observed=True)[["psbulk_n_cells", "umis_per_cell", "lib_size"]].median().round(0).astype(int)
summary.columns = ["cells per sample", "UMIs per cell", "library size"]
print("Median of the treated samples:"); display(summary)
print(f"PBS controls are {obs.loc[is_ctrl, 'lib_size'].median() / obs.loc[~is_ctrl, 'lib_size'].median():.1f}× deeper than a typical treated sample.")

#### ❓ Exercise 2.3 · Question

**Why do the four cell types have such different library sizes? And why are the PBS samples so much larger?**

<details>
<summary>💡 Answer</summary>

**Library size = cells × UMIs per cell.**
* **Number of cells.** CD4 T cells are by far the most abundant cell type in these samples (≈ 3,000 cells per sample), NK cells are rare (≈ 400). The more cells we sum, the more counts we get.
* **RNA content per cell.** Monocytes are bigger cells with more mRNA: ≈ 6,700 UMIs per cell vs ≈ 3,000 for lymphocytes. That's why monocytes reach a library size similar to CD4 T cells with only ~40 % of the cells.
* **PBS controls** are the sum of **6 wells** (~5× more cells), so their libraries are much larger.

None of this is the biology we are after, so before comparing samples we have to remove it.

</details>

## 2.4 · Normalization

### Why? An example

Imagine you want to know whether the word *"cytokine"* is used more in a 10-page article than in a 100-page review. Counting words is not enough: the review has more words of *everything*.
You'd compare **frequencies** (occurrences per 10,000 words). That is exactly what we need for genes.

Here is a toy version with **the same donor**: a PBS control (summed over 6 wells, so *deep*: 5,000 counts in total) and an IFN-γ sample (*shallow*: 1,000 counts in total).
Gene **B** is an interferon-response gene that is truly induced by IFN-γ. The other genes do not change.

In [ ]:
toy = pd.DataFrame({"PBS (deep)":       [1000, 50, 2500, 1450],
                    "IFN-γ (shallow)":  [ 200, 40,  500,  260]},
                   index=["gene A", "gene B (ISG)", "gene C", "gene D"])
toy["raw ratio IFN-γ / PBS"] = (toy["IFN-γ (shallow)"] / toy["PBS (deep)"]).round(2)
print("RAW COUNTS   (library sizes: 5,000 vs 1,000)")
display(toy)

cpm_toy = toy.iloc[:, :2] / toy.iloc[:, :2].sum() * 1e6              # counts per million = count / library size × 1,000,000
cpm_toy["CPM ratio IFN-γ / PBS"] = (cpm_toy["IFN-γ (shallow)"] / cpm_toy["PBS (deep)"]).round(2)
print("COUNTS PER MILLION (CPM)")
display(cpm_toy.round(0))

In raw counts **every gene looks lower with IFN-γ — including the interferon-response gene B** — simply because that sample is 5× shallower.
After dividing by the library size, genes A, C, D are unchanged and gene **B is 4× up**: the right answer.

**Counts Per Million (CPM)** expresses each gene as *parts per million of the sample's molecules*:

$$\text{CPM}_{ij} \;=\; \frac{\text{count}_{ij}}{L_i}\times 10^{6},\qquad L_i=\sum_{j}\text{count}_{ij}$$

### Does it work on real data?

If CPM works, a **housekeeping** gene (`ACTB`) should *not* depend on the library size any more. Monocytes, all conditions (the deep PBS samples included):

In [ ]:
counts = adata.X.copy()                                  # raw counts, kept aside
lib    = counts.sum(axis=1, keepdims=True)               # library size of every sample (a column of 4,224 numbers)
cpm    = counts / lib * 1e6                              # counts per million
print("Every sample now sums to one million:", np.allclose(cpm.sum(axis=1), 1e6, rtol=1e-4))

m_mono = (adata.obs["cell_type"] == "CD14_Mono").values
j = adata.var_names.get_loc("ACTB")
fig, axes = plt.subplots(1, 2, figsize=(10, 3.8), sharex=True)
for ax, vals, label in zip(axes, [counts[m_mono, j], cpm[m_mono, j]], ["raw count", "CPM"]):
    ax.scatter(lib[m_mono, 0], vals, s=7, alpha=0.5, color=CT_COLORS["CD14_Mono"], linewidths=0)
    ax.set_xscale("log"); ax.set_yscale("log")
    r = np.corrcoef(np.log10(lib[m_mono, 0]), np.log10(vals + 1))[0, 1]
    ax.set_title(f"ACTB — {label}\ncorrelation with library size: r = {r:+.2f}")
    ax.set_xlabel("library size of the sample"); ax.set_ylabel(label)
plt.tight_layout(); plt.show()

Before: the counts of `ACTB` grow with the library size. After CPM: the cloud is flat — the gene is "constant", as it should be.

### The logarithm: one more problem, across *genes*

CPM fixes the **samples**. But look at the **genes**: a few are expressed at tens of thousands of CPM, most at a handful.
Statistics and PCA are driven by **variance**, so let's see how the variance of a gene relates to its expression level:

In [ ]:
log_cpm = np.log2(cpm + 1)                                                  # log2(CPM + 1)

mean_cpm, var_cpm = cpm.mean(axis=0, dtype=np.float64),     cpm.var(axis=0, dtype=np.float64)
mean_log, var_log = log_cpm.mean(axis=0, dtype=np.float64), log_cpm.var(axis=0, dtype=np.float64)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
ok = mean_cpm > 0
axes[0].hist(np.log10(mean_cpm[ok]), bins=70, color="#2a78d6")
axes[0].set_xlabel("log10(mean CPM of the gene)"); axes[0].set_ylabel("number of genes")
axes[0].set_title("① Genes span 6 orders of magnitude")

ok = (mean_cpm > 0.01) & (var_cpm > 0)
axes[1].scatter(mean_cpm[ok], var_cpm[ok], s=3, alpha=0.25, color="#2a78d6", linewidths=0, rasterized=True)
xx = np.array([0.01, 1e5]); axes[1].plot(xx, xx ** 2 * 0.5, color="k", ls="--", lw=1.2, label="reference: slope 2 (variance ∝ mean²)")
axes[1].set_xscale("log"); axes[1].set_yscale("log"); axes[1].legend(loc="upper left")
axes[1].set_xlabel("mean CPM"); axes[1].set_ylabel("variance of CPM"); axes[1].set_title("② CPM: variance rises steeply with expression")

ok = var_log > 0
axes[2].scatter(mean_log[ok], var_log[ok], s=3, alpha=0.25, color="#2a78d6", linewidths=0, rasterized=True)
axes[2].set_xlabel("mean log2(CPM + 1)"); axes[2].set_ylabel("variance of log2(CPM + 1)")
axes[2].set_title("③ log2(CPM + 1): variance no longer tied to expression")
plt.tight_layout(); plt.show()

top20_cpm = np.sort(var_cpm)[-20:].sum() / var_cpm.sum()
top20_log = np.sort(var_log)[-20:].sum() / var_log.sum()
print(f"Share of the TOTAL variance held by the 20 most variable genes:  CPM: {top20_cpm:.0%}   →   log2(CPM+1): {top20_log:.1%}")

In CPM, the variance rises steeply with the mean — roughly with its **square** for abundant genes (panel ②): a handful of very abundant genes would decide every variance-based analysis (PCA, clustering, t-tests).
The **logarithm** compresses large values, so the variance no longer depends on how abundant the gene is (panel ③):

$$\text{log-CPM}_{ij}=\log_2\bigl(\text{CPM}_{ij}+1\bigr)$$

Bonus: log₂ makes **fold changes symmetric** — doubling is +1, halving is −1 — and the *difference* of two log-values **is** the log fold change:

| fold change | ×4 | ×2 | ×1.41 | ×1 | ×0.71 | ×0.5 | ×0.25 |
|---|---|---|---|---|---|---|---|
| **log₂ fold change** | +2 | +1 | +0.5 | 0 | −0.5 | −1 | −2 |

**Why "+ 1"?** Because log(0) does not exist, and because tiny numbers give wild ratios:

In [ ]:
ex = pd.DataFrame({"gene": ["almost silent", "silent → detected", "moderate", "high"],
                   "CPM control": [0.1, 0.0, 20.0, 500.0], "CPM treated": [0.4, 0.5, 40.0, 1000.0]})
with np.errstate(divide="ignore"):
    ex["log2FC without +1"] = np.log2(ex["CPM treated"] / ex["CPM control"]).round(2)
ex["log2FC with +1"] = np.log2((ex["CPM treated"] + 1) / (ex["CPM control"] + 1)).round(2)
display(ex)

Without the pseudocount, a gene going from 0.1 to 0.4 CPM (a difference of *one or two molecules*) looks 4× up and a gene starting at zero gives infinity.
With `+1` those fold changes shrink towards 0, while moderate and high genes are unaffected: *"I don't trust ratios of tiny numbers"*.

### Apply it to the whole object

From now on `adata.X` holds **log₂(CPM + 1)**. The raw counts are not lost: they live in **layers** (more matrices with the same shape as `X`).

In [ ]:
adata.layers["counts"] = counts            # raw counts
adata.layers["cpm"]    = cpm               # counts per million
adata.X                = log_cpm           # log2(CPM + 1): the default matrix from now on
print(adata)
print(f"\nX now ranges from {adata.X.min():.1f} to {adata.X.max():.1f}  (the raw counts went up to {adata.layers['counts'].max():,.0f})")

## 2.5 · Filtering lowly expressed genes

Many of our 18,120 genes are barely detected in blood cells. Keeping them is a problem because:

1. their counts are almost **pure sampling noise** — they carry no information;
2. their fold changes are **unreliable**: with 2 molecules vs 4 molecules, a "doubling" is just bad luck;
3. every extra gene is **one more statistical test** to correct for in Part 4 (more tests → less power).

### A clear example: the same gene, measured twice

We simulate two **replicates of the same gene** (no true change!), at four expression levels. Counts follow a Poisson distribution, the minimum noise any counting process has.

In [ ]:
# 🔧 CHANGE ME: other expression levels (expected counts), e.g. [1, 5, 50, 500]
true_means = [2, 20, 200, 2000]                                    # expected number of counts of the gene
fig, axes = plt.subplots(1, 4, figsize=(14, 2.9), sharex=True, sharey=True)
for ax, lam in zip(axes, true_means):
    rep1, rep2 = rng.poisson(lam, 5000), rng.poisson(lam, 5000)    # two replicates, 5,000 "genes" with the same expression
    lfc = np.log2((rep1 + 1) / (rep2 + 1))                         # fold change between replicates: the truth is 0
    ax.hist(lfc, bins=np.linspace(-4, 4, 41), color="#2a78d6" if lam >= 200 else PBS_COLOR)
    ax.set_title(f"expected count = {lam}\n{np.mean(np.abs(lfc) > 1):.0%} of genes with |log2FC| > 1", fontsize=9.5)
    ax.set_xlabel("log2 fold change between replicates")
axes[0].set_ylabel("number of genes")
plt.tight_layout(); plt.show()

At 2 expected counts, **a large fraction of genes show a >2-fold "change" out of pure chance**; at 2,000 counts, none. Low-count genes produce **false fold changes**.

### The rule

We keep a gene if it has **at least 10 counts in at least 12 samples**: 12 is the number of donors, i.e. the size of one experimental group.
A gene that never reaches that in a whole group cannot be tested anywhere. (This is the idea behind `filterByExpr` of the *edgeR* package.)

In [ ]:
# 🔧 CHANGE ME: a stricter or looser filter, and see how many genes survive
MIN_COUNT, MIN_SAMPLES = 10, 12          # ≥ 10 counts in ≥ 12 samples

def is_expressed(counts, min_count=MIN_COUNT, min_samples=MIN_SAMPLES):
    '''For a samples × genes matrix of raw counts: True for the genes with at least `min_count` counts in at least `min_samples` samples.'''
    return (counts >= min_count).sum(axis=0) >= min_samples

keep = is_expressed(adata.layers["counts"])

n_ok = (adata.layers["counts"] >= MIN_COUNT).sum(axis=0)           # for each gene: in how many samples does it have ≥ 10 counts?
fig, ax = plt.subplots(figsize=(7.5, 3.6))
ax.hist(np.log10(n_ok + 1), bins=60, color="#2a78d6")
ax.axvline(np.log10(MIN_SAMPLES + 1), color=UP_COLOR, ls="--"); ax.text(np.log10(MIN_SAMPLES + 1) - 0.05, ax.get_ylim()[1] * 0.9, "threshold: 12 samples", color=UP_COLOR, ha="right")
ax.set_xlabel("log10(1 + number of samples with ≥ 10 counts)"); ax.set_ylabel("number of genes")
ax.set_title(f"{keep.sum():,} of {len(keep):,} genes pass the filter")
plt.tight_layout(); plt.show()

removed = adata.var_names[~keep]
print(f"Removed genes: {len(removed):,}   (they carry only {adata.layers['counts'][:, ~keep].sum() / adata.layers['counts'].sum():.3%} of all counts)")
print("A few of them:", list(rng.choice(removed, 10, replace=False)))

The removed genes are essentially **absent** in blood cells (olfactory receptors, genes of other tissues, …) and hold almost none of the counts: removing them loses nothing. Now we apply the filter:

In [ ]:
adata = adata[:, keep].copy()            # subset the genes; X, layers and var are cut together
print(adata)

## 2.6 · Heatmaps: before and after normalizing

A heatmap shows many genes (rows) in many samples (columns). We will draw the **same genes** in the **same samples**, twice:
with the **raw counts** and with the **normalized values**, and watch what normalization changes.

**Samples (columns):** the 4 cell types × 5 conditions (the PBS control + 4 cytokines that trigger very different programmes) × 12 donors = 240 samples.
The grey bar on top shows the **library size** of every column.

**One more trick, only for heatmaps: the z-score.** Every gene has its own baseline level, so colouring raw values would just show which genes are abundant.
We therefore **scale every gene (row) across the samples**: subtract its mean and divide by its standard deviation, so that

$$z_{gi}=\frac{x_{gi}-\bar{x}_{g}}{s_{g}}\qquad\Rightarrow\qquad \text{red = above this gene's average, blue = below}$$

In [ ]:
#@title Heatmap helper (run once, no need to read) { display-mode: "form" }
CONDITIONS = ["PBS", "IFN-gamma", "IL-4", "IL-1-beta", "IL-2"]

def zscore_rows(M):
    '''z-score every ROW (gene) across its columns (samples).'''
    return (M - M.mean(axis=1, keepdims=True)) / M.std(axis=1, ddof=1, keepdims=True)

def before_after_heatmap(adata, gene_sets, conditions=CONDITIONS, title=""):
    cols   = [f"{d}|{ct}|{cy}" for ct in CELL_TYPES for cy in conditions for d in DONORS]      # ordered: cell type > condition > donor
    genes  = [g for gs in gene_sets.values() for g in gs if g in adata.var_names]
    sub    = adata[cols, genes]
    panels = [("BEFORE: raw counts", sub.layers["counts"].T), ("AFTER: log2(CPM + 1)", sub.X.T)]
    lib    = np.log10(adata.obs.loc[cols, "lib_size"].values)[None, :]                         # library size of every column
    ncol   = len(conditions) * len(DONORS)                                                      # columns per cell type
    sizes  = [len([g for g in gs if g in adata.var_names]) for gs in gene_sets.values()]

    fig = plt.figure(figsize=(14, 0.26 * len(genes) + 2.6))
    gs  = fig.add_gridspec(3, 3, height_ratios=[0.9, 0.9, len(genes)], width_ratios=[1, 1, 0.025],
                           hspace=0.05, wspace=0.07, top=0.88, bottom=0.14, left=0.06, right=0.95)
    for c, (name, M) in enumerate(panels):
        ax_ct, ax_lib, ax = (fig.add_subplot(gs[r, c]) for r in range(3))
        # top bar: cell types
        for k, ct in enumerate(CELL_TYPES):
            ax_ct.add_patch(plt.Rectangle((k * ncol - 0.5, 0), ncol, 1, color=CT_COLORS[ct], lw=0))
            ax_ct.text(k * ncol + ncol / 2 - 0.5, 0.5, CT_SHORT[ct], ha="center", va="center", fontsize=9, fontweight="bold",
                       color="black" if ct == "NK_CD56low" else "white")
        ax_ct.set_xlim(-0.5, len(cols) - 0.5); ax_ct.set_ylim(0, 1); ax_ct.axis("off")
        ax_ct.set_title(name, fontsize=11.5, pad=8)
        # second bar: library size of each sample (dark = deep)
        ax_lib.imshow(lib, aspect="auto", cmap="Greys", interpolation="nearest")
        ax_lib.set_xticks([]); ax_lib.set_yticks([0] if c == 0 else [])
        if c == 0: ax_lib.set_yticklabels(["library size"], fontsize=7.5)
        for sp in ax_lib.spines.values(): sp.set_visible(False)
        # the heatmap: z-score of every gene across the samples
        im = ax.imshow(zscore_rows(M), aspect="auto", cmap=DIVERGING, vmin=-2.5, vmax=2.5, interpolation="nearest")
        ax.set_yticks(range(len(genes))); ax.set_yticklabels(genes if c == 0 else [], fontsize=8)
        ax.set_xticks([(k + 0.5) * len(DONORS) - 0.5 for k in range(len(cols) // len(DONORS))])
        ax.set_xticklabels(list(conditions) * len(CELL_TYPES), rotation=90, fontsize=7)
        for b in np.cumsum(sizes)[:-1] - 0.5: ax.axhline(b, color="white", lw=2)                # gene groups
        for k in range(1, len(CELL_TYPES)): ax.axvline(k * ncol - 0.5, color="white", lw=2)      # cell types
        for sp in ax.spines.values(): sp.set_visible(False)
    fig.colorbar(im, cax=fig.add_subplot(gs[2, 2]), label="z-score of the gene")
    fig.suptitle(title, y=0.985, fontweight="bold", fontsize=11)
    plt.show()

### Cell-type markers

In [ ]:
# 🔧 CHANGE ME: add or replace genes (any gene symbol of the dataset)
CELL_TYPE_MARKERS = {
    "monocytes": ["LYZ", "CD14", "S100A9", "FCN1"],
    "T cells":   ["CD3E", "CD3D"],
    "CD8 T":     ["CD8A", "CD8B"],
    "NK cells":  ["NKG7", "GNLY", "KLRF1"],
}
before_after_heatmap(adata, CELL_TYPE_MARKERS, title="Cell-type markers  ·  rows: genes, columns: 240 samples (12 donors × 5 conditions × 4 cell types)")

#### ❓ Exercise 2.6a · Question

**Look at the grey bar (library size: dark = deep) and compare the two panels.**
**BEFORE:** which columns are the brightest, whatever the gene? **AFTER:** is that still true? Can you now read the four cell types?

<details>
<summary>💡 Answer</summary>

* **Before**, the brightest columns are always the **PBS** ones (the dark ones in the library-size bar), for every cell type: the picture shows *how many counts each sample has*, not what the cells are, and the other conditions of the same cell type look pale.
* **After**, the PBS glow is gone. Every cell type shows a uniform pattern across its five conditions and its own markers: `LYZ`, `CD14`, `S100A9`, `FCN1` in monocytes, `CD3E`/`CD3D` in T cells, `CD8A`/`CD8B` in CD8 T cells, `NKG7`/`GNLY`/`KLRF1` in NK cells.

</details>

### Genes of the immune response

Four cytokines, four signalling pathways, four sets of target genes. Which samples light up which genes?

In [ ]:
# 🔧 CHANGE ME: add your favourite immune genes; the 5 conditions to show are CONDITIONS (hidden helper above)
IMMUNE_GENES = {
    "IFN-γ / STAT1":     ["STAT1", "IRF1", "GBP1", "IDO1", "CXCL9", "CXCL10"],
    "IL-4 / STAT6":      ["ALOX15", "CCL17", "FCER2"],
    "IL-1β / NF-κB":     ["IL6", "CXCL8", "CCL2"],
    "IL-2 / STAT5":      ["IL2RA", "GZMB", "CISH"],
}
before_after_heatmap(adata, IMMUNE_GENES, title="Immune-response genes  ·  PBS and four cytokines, in the four cell types")

#### ❓ Exercise 2.6b · Question

**Which cytokine switches on which block of genes? In which cell type? Is any response shared by several cell types?**

<details>
<summary>💡 Answer</summary>

* **IL-4** → `ALOX15`, `CCL17`, `FCER2`: almost only in **monocytes**. (IL-4 also *lowers* the interferon genes: the blue IL-4 block in the top rows.)
* **IFN-γ** → the interferon-stimulated genes (`GBP1`, `IDO1`, `CXCL9`, `CXCL10`…), strongest in **monocytes**. `STAT1`, `IRF1` and `GBP1` rise in T and NK cells too: a *shared* core.
* **IL-1β** → inflammatory genes (`IL6`, `CXCL8`, `CCL2`) in **monocytes**. IL-1β and IL-2 also raise interferon genes in monocytes, probably through secondary cytokine release (the paper makes this point for IL-2).
* **IL-2** → `IL2RA`, `GZMB`, `CISH` in **NK and T cells**, the lymphocytes that carry the IL-2 receptor.

The normalized heatmap shows **biology**; the raw one shows **sequencing depth**.

</details>

> ⚠️ **z-scores are for visualization** (and clustering). They throw away absolute levels and exaggerate genes that are flat but noisy.
> Never compute fold changes from z-scores — in Part 4 we use differences of log₂(CPM + 1).

## 2.7 · Recap

| Step | What | Fixes | Stored in |
|:--:|---|---|---|
| 1 | **CPM** = count / library size × 10⁶ | different library sizes (**between samples**) | `adata.layers["cpm"]` |
| 2 | **log₂(CPM + 1)** | huge dynamic range and mean–variance link (**across genes**) | `adata.X` |
| 3 | **filter** genes (≥ 10 counts in ≥ 12 samples) | noise and too many tests | rows of `adata` |
| 4 | **z-score per gene** | baselines hiding the pattern (**in heatmaps only**) | not stored |

`adata.layers["counts"]` = raw counts → `adata.layers["cpm"]` = CPM → `adata.X` = log₂(CPM + 1).

☕ **Break — 10 minutes.**

---
# Part 3 · Principal Component Analysis (PCA): one analysis to rule them all
⏱ *35 min* (≈ 10 min of slides + 25 min here) — **Goal:** look at thousands of genes at once and find out **what drives the variation** between samples.

Each of our 4,224 samples is described by ~15,500 numbers (one per gene): impossible to plot. **PCA** finds a *few new axes* that keep as much of the variation between samples as possible.
It is **unsupervised** (it knows nothing about cytokines, donors or cell types) so, to find out what a PC *means*, we **colour the samples by metadata** and see which colouring lines up with the axes.

| Step | What | Time |
|:--:|---|:--:|
| **3.1** | The principle with 2 genes: **rotation, SVD, projection** | 6 min |
| **3.2** | PCA of **all the samples** | 4 min |
| **3.3** | One plot, **four colourings**: cell type, donor, cytokine, control vs treated | 5 min |
| **3.4** | What does each PC mean? **R²** and **loadings** | 4 min |
| **3.5** | **Zoom in** on one cell type | 4 min |
| **3.6** | 🔧 Your turn | 2 min |

In [ ]:
#@title 🏃 Catch-up cell: tick the box ONLY if Part 2 did not run to the end (it rebuilds `adata` in one go) { display-mode: "form" }
REBUILD_ADATA = False  #@param {type:"boolean"}

if REBUILD_ADATA:
    MIN_COUNT, MIN_SAMPLES = 10, 12
    def is_expressed(counts, min_count=MIN_COUNT, min_samples=MIN_SAMPLES):
        return (counts >= min_count).sum(axis=0) >= min_samples

    # tidy (2.1)
    adata = adata_raw[~((adata_raw.obs["cytokine"] == "PBS") & adata_raw.obs["bc1_well"].notna()).values].copy()
    if "psbulk_props" in adata.layers:
        del adata.layers["psbulk_props"]
    obs = adata.obs[["donor", "cytokine", "treatment", "cell_type", "psbulk_n_cells"]].copy()
    cytokines = ["PBS"] + sorted(c for c in obs["cytokine"].astype(str).unique() if c != "PBS")
    obs["donor"]     = pd.Categorical(obs["donor"].astype(str), categories=DONORS)
    obs["cytokine"]  = pd.Categorical(obs["cytokine"].astype(str), categories=cytokines)
    obs["cell_type"] = pd.Categorical(obs["cell_type"].astype(str), categories=CELL_TYPES)
    obs["treatment"] = pd.Categorical(np.where(obs["cytokine"] == "PBS", "control", "treated"), categories=["control", "treated"])
    obs.index = (obs["donor"].astype(str) + "|" + obs["cell_type"].astype(str) + "|" + obs["cytokine"].astype(str)).values
    adata.obs = obs
    X = adata.X.toarray() if hasattr(adata.X, "toarray") else np.asarray(adata.X)
    adata.X = X.astype(np.float32)
    adata.var = pd.DataFrame(index=adata.var_names)
    adata.var["total_counts"]       = adata.X.sum(axis=0)
    adata.var["n_samples_detected"] = (adata.X > 0).sum(axis=0)
    # library sizes, CPM, log2(CPM + 1) (2.3-2.4)
    counts = adata.X.copy()
    lib = counts.sum(axis=1, keepdims=True)
    cpm = counts / lib * 1e6
    adata.obs["lib_size"]      = lib[:, 0]
    adata.obs["umis_per_cell"] = adata.obs["lib_size"] / adata.obs["psbulk_n_cells"]
    adata.layers["counts"], adata.layers["cpm"] = counts, cpm
    adata.X = np.log2(cpm + 1)
    # filter (2.5)
    adata = adata[:, is_expressed(adata.layers["counts"])].copy()
    print("✅ adata rebuilt:", adata.shape)
else:
    print("Nothing to do: we use the `adata` from Part 2.")

### Settings of this part

In [ ]:
# 🔧 CHANGE ME: all the parameters of this part
N_HVG = 2000                                                                       # number of highly variable genes used for PCA
PCA_CELL_TYPE = "CD14_Mono"                                                        # cell type of the zoom-in of 3.5: "CD4_T_cell", "CD8_T_cell", "NK_CD56low" or "CD14_Mono"
CYTOKINES_PANEL = ["IFN-gamma", "IL-4", "TNF-alpha", "IL-10", "IL-6", "GM-CSF"]   # cytokines highlighted in the plots (any of the 87; up to 7)
assert len(CYTOKINES_PANEL) <= 7, "At most 7 cytokines (colours get hard to tell apart)"
print(adata)

In [ ]:
#@title PCA plotting helpers (run once, no need to read) { display-mode: "form" }
DONOR_HUES  = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#6250d6"]      # 12 donors = 6 hues × 2 marker shapes
CYT_HUES    = DONOR_HUES + ["#008300"]
CYT_MARKERS = ["o", "s", "^", "D", "v", "P", "X"]
SEQ_POINTS  = LinearSegmentedColormap.from_list("seqp", ["#cde2fb", "#2a78d6", "#0d366b"])

def pca_scatter(ax, data, color_by, pcs=(0, 1), s=14, title=None):
    '''Scatter plot of data.obsm["X_pca"], coloured by "cell_type", "donor", "cytokine", "treatment", a numeric column of data.obs or a gene name.'''
    sc, (a, b), var, obs = data.obsm["X_pca"], pcs, data.uns["pca_var"], data.obs
    kw = dict(linewidths=0.3, edgecolors="white")
    if color_by == "cell_type":
        for ct in CELL_TYPES:
            m = (obs["cell_type"] == ct).values
            if m.any(): ax.scatter(sc[m, a], sc[m, b], s=s, alpha=0.8, color=CT_COLORS[ct], marker=CT_MARKERS[ct], label=CT_LABEL[ct], **kw)
    elif color_by == "donor":
        for i, d in enumerate(DONORS):
            m = (obs["donor"] == d).values
            ax.scatter(sc[m, a], sc[m, b], s=s, alpha=0.85, color=DONOR_HUES[i % 6], marker="o" if i < 6 else "^", label=d, **kw)
    elif color_by == "cytokine":
        m = ~obs["cytokine"].isin(["PBS"] + CYTOKINES_PANEL).values
        ax.scatter(sc[m, a], sc[m, b], s=s * 0.6, color=NS_COLOR, alpha=0.5, linewidths=0, label="all other cytokines", zorder=1)
        for i, cy in enumerate(CYTOKINES_PANEL):
            m = (obs["cytokine"] == cy).values
            ax.scatter(sc[m, a], sc[m, b], s=s * 1.4, color=CYT_HUES[i], marker=CYT_MARKERS[i], label=cy, zorder=2, **kw)
        m = (obs["cytokine"] == "PBS").values
        ax.scatter(sc[m, a], sc[m, b], s=s * 2.4, color="black", label="PBS", zorder=3, **kw)
    elif color_by == "treatment":
        m = (obs["treatment"] == "treated").values
        ax.scatter(sc[m, a], sc[m, b], s=s * 0.8, color=NS_COLOR, alpha=0.6, linewidths=0, label=f"treated (cytokine): {m.sum():,}", zorder=1)
        ax.scatter(sc[~m, a], sc[~m, b], s=s * 2.4, color="black", label=f"control (PBS): {(~m).sum():,}", zorder=2, **kw)
    else:                                                                               # a number: a column of obs, or the expression of a gene
        if color_by in obs.columns: values, label = obs[color_by].values.astype(float), color_by
        else:                       values, label = np.asarray(data[:, color_by].X).ravel(), f"{color_by}\nlog2(CPM+1)"
        order = np.argsort(values)                                                      # highest values drawn on top
        pts = ax.scatter(sc[order, a], sc[order, b], c=values[order], cmap=SEQ_POINTS, s=s * 1.6, **kw)
        plt.colorbar(pts, ax=ax, shrink=0.8, pad=0.01, label=label)
    if color_by in ("cell_type", "donor", "cytokine", "treatment"):
        ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1.0), fontsize=7.5, markerscale=1.5, handletextpad=0.3, labelspacing=0.3,
                  ncol=2 if color_by == "donor" else 1)
    ax.set_xlabel(f"PC{a + 1} ({var[a]:.1f} %)"); ax.set_ylabel(f"PC{b + 1} ({var[b]:.1f} %)")
    ax.set_title(title or f"coloured by {color_by.replace('_', ' ')}")

def pca_four_views(data, pcs=(0, 1), title="", views=("cell_type", "donor", "cytokine", "treatment")):
    '''The same PCA coordinates, coloured four ways.'''
    fig, axes = plt.subplots(2, 2, figsize=(14, 10.5))
    for ax, cb in zip(axes.ravel(), views):
        pca_scatter(ax, data, cb, pcs=pcs)
    fig.suptitle(title, fontweight="bold", y=0.995)
    plt.tight_layout(); plt.show()

def r2_categorical(y, groups):
    '''Fraction of the variance of y explained by group membership (one-way ANOVA R²).'''
    y, g = np.asarray(y, float), np.asarray(groups).astype(str)
    ss_tot = ((y - y.mean()) ** 2).sum()
    ss_between = sum(((y[g == k].mean() - y.mean()) ** 2) * (g == k).sum() for k in np.unique(g))
    return ss_between / ss_tot

def pc_r2_table(data, n_pc=8):
    '''R² of every PC with every sample variable.'''
    sc, obs = data.obsm["X_pca"], data.obs
    loglib = np.log10(obs["lib_size"].astype(float).values)
    return pd.DataFrame({
        "cell type": [r2_categorical(sc[:, k], obs["cell_type"]) for k in range(n_pc)],
        "donor":     [r2_categorical(sc[:, k], obs["donor"]) for k in range(n_pc)],
        "cytokine":  [r2_categorical(sc[:, k], obs["cytokine"]) for k in range(n_pc)],
        "control vs treated": [r2_categorical(sc[:, k], obs["treatment"]) for k in range(n_pc)],
        "library size": [np.corrcoef(sc[:, k], loglib)[0, 1] ** 2 for k in range(n_pc)],
    }, index=[f"PC{k + 1}" for k in range(n_pc)])

def plot_pc_summary(data, n_pc=8):
    '''Variance explained by each PC (left) and R² of each PC with each variable (right).'''
    var, R2 = data.uns["pca_var"][:n_pc], pc_r2_table(data, n_pc)
    fig, axes = plt.subplots(1, 2, figsize=(11.5, 3.9), gridspec_kw={"width_ratios": [1, 1.6]})
    axes[0].barh(range(n_pc), var, color="#2a78d6"); axes[0].invert_yaxis()
    axes[0].set_yticks(range(n_pc)); axes[0].set_yticklabels(R2.index); axes[0].set_xlabel("variance explained (%)"); axes[0].set_title("How much variance?")
    for i, v in enumerate(var): axes[0].text(v + 0.5, i, f"{v:.1f}", va="center", fontsize=8)
    im = axes[1].imshow(R2.values, cmap=SEQUENTIAL, vmin=0, vmax=1, aspect="auto")
    for i in range(R2.shape[0]):
        for j in range(R2.shape[1]):
            axes[1].text(j, i, f"{R2.values[i, j]:.2f}", ha="center", va="center", fontsize=8.5, color="w" if R2.values[i, j] > 0.55 else "k")
    axes[1].set_xticks(range(R2.shape[1])); axes[1].set_xticklabels(R2.columns, rotation=20, ha="right"); axes[1].set_yticks(range(n_pc)); axes[1].set_yticklabels(R2.index)
    axes[1].set_title("What does each PC capture?  (R² with each variable)"); plt.colorbar(im, ax=axes[1], shrink=0.85)
    plt.tight_layout(); plt.show()

def plot_loadings(data, loadings, genes, pcs=(0, 1, 2), n=8):
    '''The genes with the largest positive and negative weights of each PC.'''
    fig, axes = plt.subplots(1, len(pcs), figsize=(4.6 * len(pcs), 4.3))
    for ax, k in zip(np.atleast_1d(axes), pcs):
        s = pd.Series(loadings[:, k], index=genes).sort_values(); top = pd.concat([s.head(n), s.tail(n)])
        ax.barh(top.index, top.values, color=[DOWN_COLOR if v < 0 else UP_COLOR for v in top.values])
        ax.axvline(0, color="k", lw=0.6); ax.tick_params(axis="y", labelsize=8)
        ax.xaxis.set_major_locator(plt.MaxNLocator(4)); ax.set_xlabel("loading"); ax.set_title(f"PC{k + 1}  ({data.uns['pca_var'][k]:.1f} % of the variance)")
    plt.tight_layout(); plt.show()

## 3.1 · The principle with 2 genes: rotation, SVD and projection

Take the **monocytes** and just **two genes**, `ISG15` and `MX1` (both interferon-stimulated genes). Every sample is a point in a plane: *x* = `ISG15`, *y* = `MX1`.
The two genes go up and down together, so the cloud is **elongated**. PCA finds that direction, in three steps (the slides showed the picture):

1. **Centre** every gene (subtract its mean): matrix $X_c$.
2. **SVD** (singular value decomposition): $X_c = U\,S\,V^\top$. $V$ is a **rotation**: its columns (the *loadings*) are the new axes PC1, PC2 — unit length, perpendicular. $S$ holds how much the data is stretched along each axis.
3. **Projection**: the coordinates of every sample on the new axes, the *scores*, are $X_c V$ (which equals $U S$).

In [ ]:
is_mono = (adata.obs["cell_type"] == "CD14_Mono").values
P = adata[is_mono, ["ISG15", "MX1"]].X.astype(np.float64)       # 1,056 monocyte samples × 2 genes, in log2(CPM + 1)

Pc = P - P.mean(axis=0)                                         # 1) centre every gene
U, S, Vt = np.linalg.svd(Pc, full_matrices=False)               # 2) SVD:  Pc = U · diag(S) · Vt
if Vt[0].sum() < 0: Vt[0], U[:, 0] = -Vt[0], -U[:, 0]           #    the sign of a PC is arbitrary: we make PC1 point to "more of both genes"
V = Vt.T                                                        #    columns of V = PC1, PC2 (the loadings)
scores = Pc @ V                                                 # 3) projection of every sample on the new axes

var_pct = S ** 2 / np.sum(S ** 2) * 100                         #    variance carried by each axis
print("Loadings V  (columns = PC1, PC2;  rows = ISG15, MX1):\n", np.round(V, 3))
print(f"\nPC1 is the x axis rotated by {np.degrees(np.arctan2(V[1, 0], V[0, 0])):.0f}°  |  V is a pure rotation (VᵀV = identity): {np.allclose(V.T @ V, np.eye(2))}")
print(f"Scores = U · S: {np.allclose(scores, U * S)}")
print(f"Variance explained:  PC1 = {var_pct[0]:.1f} %    PC2 = {var_pct[1]:.1f} %")

In [ ]:
is_ifn = adata.obs.loc[is_mono, "cytokine"].astype(str).str.startswith("IFN-").values           # the 8 interferons
mu = P.mean(axis=0)
fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))

# (left) the original axes, with the two new axes drawn on top, and the projection of a few samples on PC1
ax = axes[0]
for flag, col, lab in [(False, NS_COLOR, "other cytokines and PBS"), (True, UP_COLOR, "interferons (α, β, γ, λ, ε, ω)")]:
    ax.scatter(P[is_ifn == flag, 0], P[is_ifn == flag, 1], s=9, alpha=0.65, color=col, linewidths=0, label=lab)
for i in rng.choice(len(P), 14, replace=False):                                                 # projection = drop a perpendicular onto PC1
    proj = mu + (Pc[i] @ V[:, 0]) * V[:, 0]
    ax.plot([P[i, 0], proj[0]], [P[i, 1], proj[1]], color="k", lw=0.9, ls=":")
    ax.scatter(*proj, s=14, color="k", zorder=5)
for k, nm in enumerate(["PC1", "PC2"]):                                                          # the new axes: length = ±1.5 standard deviations
    d = V[:, k] * 1.5 * S[k] / np.sqrt(len(P) - 1)
    ax.annotate("", xy=mu + d, xytext=mu - d, arrowprops=dict(arrowstyle="<|-|>", color="k", lw=1.8), annotation_clip=False)
    ax.annotate(nm, xy=mu + d, xytext=(6, 4), textcoords="offset points", fontsize=11, fontweight="bold", annotation_clip=False)
ax.set_aspect("equal"); ax.set_xlabel("ISG15  (log2 CPM + 1)"); ax.set_ylabel("MX1  (log2 CPM + 1)"); ax.legend(loc="lower right")
ax.set_title("Original axes (dotted: projection on PC1)")

# (right) the same points in the rotated axes
ax = axes[1]
for flag, col in [(False, NS_COLOR), (True, UP_COLOR)]:
    ax.scatter(scores[is_ifn == flag, 0], scores[is_ifn == flag, 1], s=9, alpha=0.65, color=col, linewidths=0)
ax.axhline(0, color="k", lw=0.6); ax.axvline(0, color="k", lw=0.6); ax.set_aspect("equal")
ax.set_xlabel(f"PC1 ({var_pct[0]:.0f} % of the variance)"); ax.set_ylabel(f"PC2 ({var_pct[1]:.0f} %)")
ax.set_title("New axes: the same points, rotated")
plt.tight_layout(); plt.show()

PCA is **just a rotation** (plus centering): nothing has been lost yet. **Dimensionality reduction** happens when we keep only the first few axes:
here **PC1 alone** summarizes almost all the variation, and both genes have a positive loading, so PC1 = *"overall interferon-gene level"* — the interferon samples (red) sit at its high end.
With thousands of genes the recipe is **exactly the same**.

## 3.2 · PCA of all the samples

With ~15,500 genes we first reduce the noise: most genes barely vary between samples. We keep the **highly variable genes (HVGs)**: the `N_HVG` genes with the largest variance of log₂(CPM + 1) across samples.
Then we run the same three steps, wrapped in a function:

In [ ]:
def pca_svd(M, n_components=10):
    '''PCA of a samples × genes matrix: centre → SVD → project. Returns scores, % of variance explained, loadings.'''
    Mc = M.astype(np.float64) - M.mean(axis=0)                  # 1) centre the genes
    U, S, Vt = np.linalg.svd(Mc, full_matrices=False)           # 2) SVD
    scores   = (U * S)[:, :n_components]                        # 3) projection: coordinates of the samples on the PCs
    var_pct  = (S ** 2 / np.sum(S ** 2) * 100)[:n_components]   #    % of the total variance carried by each PC
    loadings = Vt[:n_components].T                              #    weights of the genes in each PC (genes × PCs)
    return scores, var_pct, loadings

gene_var = adata.X.var(axis=0)
hvg      = np.argsort(gene_var)[::-1][:N_HVG]                   # indices of the most variable genes
print("The 10 most variable genes:", adata.var_names[hvg[:10]].tolist())

scores, var_pct, loadings = pca_svd(adata.X[:, hvg])
adata.obs["log10_lib_size"] = np.log10(adata.obs["lib_size"].astype(float))    # (used to colour the plots)
adata.obsm["X_pca"]  = scores                                   # AnnData stores multi-dimensional sample annotations in `obsm`
adata.uns["pca_var"] = var_pct

sk = PCA(n_components=10).fit(adata.X[:, hvg])                  # check: scikit-learn's PCA gives the same variances
print("Same as scikit-learn:", np.allclose(var_pct, sk.explained_variance_ratio_ * 100, atol=1e-3))

fig, ax = plt.subplots(figsize=(6.5, 3.4))
ax.bar(range(1, 11), var_pct, color="#2a78d6")
for i, v in enumerate(var_pct[:5]): ax.text(i + 1, v + 1, f"{v:.1f} %", ha="center", fontsize=8.5)
ax.set_xticks(range(1, 11)); ax.set_xlabel("principal component"); ax.set_ylabel("variance explained (%)")
ax.set_title(f"Scree plot: the first 4 PCs hold {var_pct[:4].sum():.0f} % of the variance")
plt.tight_layout(); plt.show()

## 3.3 · One plot, four colourings

The coordinates never change, only the colour. Samples are **cell types**, **donors**, **cytokines** (the PBS control in black, the cytokines of `CYTOKINES_PANEL` in colour, all others in grey) and
**control vs treated**.

In [ ]:
pca_four_views(adata, pcs=(0, 1), title=f"All {adata.n_obs:,} samples · PCA of the {N_HVG:,} most variable genes · PC1 vs PC2")

#### ❓ Exercise 3.3 · Questions
1. Which PCs separate the **cell types**, and how much variance do they hold? Which two cell types are the most similar?
2. Do **donors**, **cytokines** or **control vs treated** separate on PC1–PC2?
3. Does that mean that cytokines have no effect on the cells?

<details>
<summary>💡 Answers</summary>

1. **PC1 (66 %)** separates monocytes from lymphocytes; **PC2 (17 %)** separates NK cells from T cells. CD4 and CD8 T cells are the closest pair (they overlap on these two PCs).
2. **No.** Inside every cell-type cluster the donors, the cytokines and the controls are mixed; the black PBS points sit *inside* the cloud of treated samples.
3. **No!** PCA shows the **largest** sources of variation first, and the difference between a monocyte and a T cell is far bigger than the effect of any cytokine. The cytokine and donor effects live on *later* PCs, or inside each cell type (next steps).

</details>

## 3.4 · What does each PC mean?

Colours give an impression; numbers give certainty. For every PC we compute the **R²** with each variable: the fraction of the PC's variance explained by the variable
(for a category: between-group variance / total variance, as in an ANOVA; for the library size: the squared correlation).

In [ ]:
plot_pc_summary(adata, n_pc=8)

Next, a PC is a weighted sum of genes. The **loadings** tell which genes carry the largest weights, i.e. **which genes define the axis**
(the sign of a PC is arbitrary: flipping all the signs gives an equivalent PCA).

In [ ]:
plot_loadings(adata, loadings, adata.var_names[hvg], pcs=(0, 1, 2))

#### ❓ Exercise 3.4 · Questions
1. Using the R² table and the loadings: what is **PC1**? **PC2**? **PC3**? Do the genes make sense?
2. What is the first PC with some **cytokine** signal? Is it a high-variance or a low-variance PC? What does that say about the size of the cytokine effects?
3. PC2 has an R² of ~0.4 with the **library size**. Should we worry about it?

<details>
<summary>💡 Answers</summary>

1. **PC1**: monocyte genes (`FPR3`, `CYBB`, `CD36`…) vs lymphocyte genes (`CD96`, `SKAP1`, `CD247`…). **PC2**: NK genes (`NCR1`, `KLRF1`, `GNLY`, `NCAM1`) vs naive-T genes (`CCR7`, `LEF1`, `TSHZ2`). **PC3** is a *mixture*: interferon-response genes (`ISG15`, `IFIT1/2/3`, `RSAD2`) on one side, **Y-chromosome genes** (`UTY`, `USP9Y`, `KDM5D`) on the other. A cytokine-like axis and a donor/sex axis in the same PC.
2. **PC3** (R² cytokine ≈ 0.13, donor ≈ 0.72). It holds only ~3 % of the variance: the cytokine effects are **small compared with the cell-type differences**.
3. Not really: NK cell samples simply have smaller libraries (Part 2), so PC2 — which separates NK from T cells — correlates with library size. It is the *cell type* in disguise (a confounder), not a depth artefact. The normalization of Part 2 did its job.

</details>

## 3.5 · Zoom in: PCA within one cell type

The cell-type differences hide everything else. A common remedy is to analyse **one cell type at a time**: we subset the object, re-select the HVGs and run PCA again. Let's do it for the cell type of `PCA_CELL_TYPE` (by default the **CD14 monocytes**, the most responsive one; the comments below describe the default).

In [ ]:
mono = adata[adata.obs["cell_type"] == PCA_CELL_TYPE].copy()                   # a new AnnData with the 1,056 samples of one cell type
hvg_m = np.argsort(mono.X.var(axis=0))[::-1][:N_HVG]                            # HVGs of the monocytes
mono.obsm["X_pca"], mono.uns["pca_var"], load_m = pca_svd(mono.X[:, hvg_m])

pca_four_views(mono, pcs=(0, 1), title=f"{CT_LABEL[PCA_CELL_TYPE]} only · PC1 vs PC2", views=("donor", "cytokine", "treatment", "log10_lib_size"))

#### ❓ Exercise 3.5 · Question

**What changed compared with the plot of all cell types?**

<details>
<summary>💡 Answer</summary>

* **Donors** now form clear groups: the *baseline differences between people* are large within a cell type. Donor 2 stands apart (the paper notes the same for its monocytes, Fig. 2k).
* **Cytokines** still do not separate on PC1–PC2: the individual effects are smaller than the donor-to-donor differences. The grey cloud is made of 87 cytokines × 12 donors, so only the colours of the highlighted ones can be told apart. This is exactly why in Part 4 every cytokine is compared with the PBS of the **same donor** (a *paired* design).
* **Control vs treated**: the PBS samples are not different from the rest on these PCs.

</details>

The cytokine effects appear on *later* PCs. Let's look at PC3 vs PC4 and at what these PCs are made of:

In [ ]:
fig, ax = plt.subplots(figsize=(7.8, 5))
pca_scatter(ax, mono, "cytokine", pcs=(2, 3), title=f"{CT_LABEL[PCA_CELL_TYPE]} · PC3 vs PC4, coloured by cytokine")
plt.tight_layout(); plt.show()

plot_loadings(mono, load_m, mono.var_names[hvg_m], pcs=(2, 3))        # the genes behind PC3 and PC4

Now the cytokines separate: **IL-4** (orange squares) leaves the cloud towards low PC4, whose negative end holds IL-4 target genes (`ALOX15`, `WNT5B`, `FN1`); **GM-CSF** (purple crosses) goes towards high PC3 (`CD1B`, `CD1E`, `CCL24`, `CLEC5A`, `INHBA`); **IL-10** (yellow diamonds) towards low PC3.
The low end of PC3 also holds **Y-chromosome genes** (`UTY`, `KDM5D`): a cytokine axis and a sex axis in the same PC, as before.
*Each cytokine moves the samples in its own direction*, mixed with donor effects.

## 3.6 · 🔧 Your turn

Change the settings and re-run the cell: `COLOR_BY` can be `"donor"`, `"cytokine"`, `"treatment"`, `"log10_lib_size"`, or **any gene symbol** (`"STAT1"`, `"IL1B"`, `"UTY"`…). The plot uses the PCA of `PCA_CELL_TYPE` (change it in the settings cell of this part, then re-run from there).

In [ ]:
# 🔧 CHANGE ME: what to colour by, and which PCs to plot
COLOR_BY = "UTY"        # "donor" | "cytokine" | "treatment" | "log10_lib_size" | any gene symbol
PCS      = (2, 3)       # which PCs on the x and y axes, e.g. (0, 1) for PC1 vs PC2

fig, ax = plt.subplots(figsize=(7.5, 5.3))
pca_scatter(ax, mono, COLOR_BY, pcs=PCS, s=34)
plt.tight_layout(); plt.show()

> 🔧 **Tasks**
> 1. Colour by `UTY` (a Y-chromosome gene). With `PCS = (0, 1)` and with `(2, 3)`: which PC separates male from female donors? *(the paper reports 6 males and 6 females)*
> 2. Colour by `log10_lib_size`. Is there a smooth gradient along an axis? Would that be a warning sign?
> 3. Change `CYTOKINES_PANEL` (in *Settings of this part*, then re-run that cell) to six interferon-family cytokines — e.g. `["IFN-alpha1", "IFN-beta", "IFN-gamma", "IFN-lambda1", "IFN-omega", "IL-27"]` — and colour by `"cytokine"`. Which ones cluster together?
> 4. Set `N_HVG = 200` and `N_HVG = 10000` and recompute (3.5). How robust is the picture?

## 3.7 · Recap

* PCA = **rotation** of the data onto axes of decreasing variance (**SVD**); **scores** = projection of the samples, **loadings** = weights of the genes.
* It is **unsupervised**: colouring by metadata tells you what each PC means; the **R²** table and the **loadings** make it quantitative.
* Normalization decides what PCA sees (Part 2); without it, the first PCs would reflect sequencing depth.
* The strongest variation is the **cell type**; **donor** effects are next; **cytokine** effects are real but smaller, and appear within a cell type or on later PCs.
* 👉 Next: ask the question gene by gene, **paired by donor** — differential expression.

---
# Part 4 · Differential expression: treated vs untreated cells
⏱ *35 min* (≈ 10 min of slides + 25 min here) — **Goal:** find the genes that **change** when a cytokine is added, say **how sure** we are, and compare two cytokines.

PCA showed that cytokines move the samples in particular directions. Differential expression asks the question **gene by gene**:

> *For gene g, is its expression with the cytokine different from its expression with PBS — more than we would expect by chance?*

| Step | Question | Tool | Time |
|:--:|---|---|:--:|
| **4.1** | How big is the change, and is it consistent across donors? | paired differences, **log₂ fold change**, a **paired t-test** by hand | 6 min |
| **4.2** | …for all genes, properly | **DESeq2** (`pydeseq2`) on the raw counts | 10 min |
| **4.3** | We did ~450 tests… | **multiple-testing correction** (Benjamini–Hochberg, **FDR**) | 5 min |
| **4.4** | Which genes do we call *differentially expressed*? | thresholds + **volcano plot** | 4 min |
| **4.5** | How do two cytokines differ? | DESeq2 for **two cytokines** in one or two cell types | 8 min |
| **4.6** | 🔧 Your turn | | 2 min |

> If `adata` does not exist (you restarted the session), run the **catch-up cell of Part 3**.

In [ ]:
# 🔧 CHANGE ME: the comparison of sections 4.1-4.4 (any of the 87 cytokines, any of the 4 cell types)
CYTOKINE        = "IFN-gamma"      # the cytokine we compare with the PBS control
CELL_TYPE       = "CD14_Mono"      # "CD4_T_cell", "CD8_T_cell", "NK_CD56low" or "CD14_Mono"
EXAMPLE_GENE    = "CXCL9"          # a gene we expect to respond to IFN-γ (use "CCL26" for IL-4)
CONTROL_GENE    = "IL10RA"         # an immune gene that does NOT respond to IFN-γ
PADJ_THRESH     = 0.05             # significance: adjusted p-value (FDR) below this…
LFC_THRESH      = 0.5              # …and |log2 fold change| above this (0.5 ≈ a 1.4× change)

# 🔧 CHANGE ME: the comparison of section 4.5 (two cytokines, in one or two cell types)
CYTOKINE_A         = "IFN-gamma"
CYTOKINE_B         = "IL-4"
CELL_TYPES_COMPARE = ["CD14_Mono", "CD4_T_cell"]     # one or two of the four cell types

print(f"4.1-4.4: {CYTOKINE} vs PBS in {CT_LABEL[CELL_TYPE]}  ·  example gene {EXAMPLE_GENE}")
print(f"4.5    : {CYTOKINE_A} vs {CYTOKINE_B} in {[CT_LABEL[c] for c in CELL_TYPES_COMPARE]}")

In [ ]:
#@title DE plotting helpers (run once, no need to read) { display-mode: "form" }
try:
    from adjustText import adjust_text
    HAS_ADJUSTTEXT = True
except ImportError:
    HAS_ADJUSTTEXT = False
CYTOKINE_COLOR = "#eb6834"

def paired_plot(ax, x_pbs, x_cyt, gene, cytokine):
    '''One line per donor: PBS → cytokine. Red = higher with the cytokine, blue = lower.'''
    for a, b in zip(x_pbs, x_cyt):
        ax.plot([0, 1], [a, b], color=UP_COLOR if b > a else DOWN_COLOR, lw=1.2, alpha=0.7, zorder=1)
    ax.scatter(np.zeros(len(x_pbs)), x_pbs, s=30, color=PBS_COLOR, zorder=2)
    ax.scatter(np.ones(len(x_cyt)), x_cyt, s=30, color=CYTOKINE_COLOR, zorder=2)
    ax.plot([0, 1], [x_pbs.mean(), x_cyt.mean()], color="k", lw=3, marker="D", ms=7, zorder=3)
    ax.set_xticks([0, 1]); ax.set_xticklabels(["PBS", cytokine]); ax.set_xlim(-0.35, 1.35)
    ax.set_ylabel("log2(CPM + 1)"); ax.set_title(f"{gene}\nmean difference = {np.mean(x_cyt - x_pbs):+.2f}")

def volcano(ax, de, title, n_label=16, xlabel="log2 fold change"):
    '''Volcano plot: effect size (x) against evidence (y); the strongest DEGs are labelled.'''
    d = de[de["tested"]].copy(); d["nlp"] = -np.log10(d["padj"].clip(lower=1e-300))
    for k, col, size, alpha in [("ns", NS_COLOR, 6, 0.5), ("down", DOWN_COLOR, 14, 0.8), ("up", UP_COLOR, 14, 0.8)]:
        sub = d[d["direction"] == k]
        ax.scatter(sub["log2fc"], sub["nlp"], s=size, color=col, alpha=alpha, linewidths=0, rasterized=True)
    for x in (-LFC_THRESH, LFC_THRESH): ax.axvline(x, color="#999", lw=0.8, ls="--")
    ax.axhline(-np.log10(PADJ_THRESH), color="#999", lw=0.8, ls="--")
    sig = d[d["direction"] != "ns"].copy()                             # label the DEGs with the best combination of effect size and evidence
    sig["score"] = sig["nlp"] * sig["log2fc"].abs()
    ax.margins(y=0.12); x0, x1 = ax.get_xlim(); y0, y1 = ax.get_ylim()                   # keep the labels out of the two counters (top corners)
    corner = (sig["nlp"] > y0 + 0.85 * (y1 - y0)) & ((sig["log2fc"] < x0 + 0.3 * (x1 - x0)) | (sig["log2fc"] > x1 - 0.3 * (x1 - x0)))
    sig = sig[~corner]
    top = pd.concat([sig[sig["direction"] == "up"].nlargest(n_label // 2, "score"), sig[sig["direction"] == "down"].nlargest(n_label // 2, "score")])
    texts = [ax.text(r.log2fc, r.nlp, g, fontsize=7) for g, r in top.iterrows()]
    if HAS_ADJUSTTEXT and texts:
        adjust_text(texts, ax=ax, arrowprops=dict(arrowstyle="-", color="#999", lw=0.5))
    ax.annotate(f"↑ {(d['direction'] == 'up').sum():,} up", xy=(0.98, 0.98), xycoords="axes fraction", ha="right", va="top", color=UP_COLOR, fontsize=10, fontweight="bold")
    ax.annotate(f"↓ {(d['direction'] == 'down').sum():,} down", xy=(0.02, 0.98), xycoords="axes fraction", ha="left", va="top", color=DOWN_COLOR, fontsize=10, fontweight="bold")
    ax.set_xlabel(xlabel); ax.set_ylabel("−log10 adjusted p-value"); ax.set_title(title)

## 4.0 · Only the immune genes

In this version of the practical we **do not test all the genes**: the DE analysis uses only a list of **immune-system genes**. The list is the union of four gene sets (retrieved from [Enrichr](https://maayanlab.cloud/Enrichr/)):

* **GO Biological Process** *Cytokine-Mediated Signaling Pathway* (GO:0019221)
* **GO Biological Process** *T Cell Activation* (GO:0042110)
* **MSigDB Hallmark** *Interferon Alpha Response* and *Interferon Gamma Response*

That gives **542 gene symbols**; the ones that are in our table of **expressed genes** (the filter of Part 2) are kept: **506**. Each DESeq2 comparison applies the expression filter again to its 24 samples, so ≈ 450 genes are finally tested.

**Why would we do this?** (1) With ~450 tests instead of ~12,700 the multiple-testing correction is much milder (section 4.3); (2) the results are about immune biology, and the volcano plots are easier to read; (3) everything runs faster.

**⚠️ What do we lose?**
* The list is **biased**: it was built *from genes that respond to cytokines*, so the **fraction of DEGs is much higher** than in a whole-transcriptome analysis (for IFN-γ in monocytes: ~42 % here, ~21 % with all genes). Do not compare those percentages.
* Genes outside the list cannot be found, even classic ones (`ACTB`, `GBP1`, `ALOX15`, `NKG7`… are not in it).
* DESeq2's normalization assumes that *most genes do not change*; in a selection full of cytokine-responsive genes this assumption is weaker (see the questions of 4.2).

In [ ]:
#@title The immune gene list (542 gene symbols; run once, no need to read) { display-mode: "form" }
IMMUNE_GENES = """
ACTN4 ADAR ADIPOQ ADIPOR1 ADIPOR2 AIM2 AKT1 APOL6 APPL1 APPL2 ARID5B ARL4A AUTS2 B2M BANK1 BATF2 BIRC2 BIRC3
BPGM BST2 BTG1 C1R C1S CARD14 CASP1 CASP3 CASP4 CASP7 CASP8 CBL CCL1 CCL11 CCL13 CCL14 CCL15 CCL16 CCL18 CCL19
CCL2 CCL21 CCL22 CCL23 CCL24 CCL25 CCL26 CCL3 CCL3L1 CCL4 CCL4L1 CCL5 CCL7 CCL8 CCM2 CCR1 CCR2 CCR7 CCRL2 CD1C
CD2 CD247 CD27 CD274 CD276 CD28 CD300LF CD38 CD3D CD3E CD4 CD40 CD44 CD47 CD69 CD7 CD70 CD74 CD80 CD84 CD86
CD8A CD8B CDKN1A CEACAM1 CEACAM21 CEBPA CFB CFH CHD7 CHUK CIITA CISH CLCF1 CLEC7A CLSTN1 CMKLR1 CMPK2 CMTR1
CNP CNTF CNTFR COMMD7 CRLF1 CRLF2 CRTAM CSF1 CSF1R CSF2 CSF2RA CSF2RB CSF3 CSF3R CSNK2B CTPS1 CTR9 CTSG CUL4A
CX3CL1 CXADR CXCL10 CXCL11 CXCL12 CXCL9 CXCR2 DCAF12 DDOST DDX58 DDX60 DHX58 DLL4 DPP4 DUOX1 DUOX2 EBI3 EDA
EDA2R EDAR EDN2 EGR1 EIF2AK2 EIF2AK4 EIF4E3 EIF5AL1 ELF1 EPOR EPSTI1 EPX EREG F2RL1 F3 FAS FCER1G FCGR1A FER
FGL2 FKBP1A FLT3 FOXC1 FOXO3 FOXP3 FPR1 GATA3 GBP2 GBP4 GBP6 GCH1 GH1 GHR GMPR GPR18 GPR35 GPR75 GPR89A GRB2
GREM2 GZMA HAX1 HCK HCLS1 HELZ2 HERC6 HIF1A HLA-A HLA-B HLA-C HLA-DMA HLA-DQA1 HLA-DRB1 HLA-G HSPD1 ICAM1
ICOSLG IDO1 IFI27 IFI30 IFI35 IFI44 IFI44L IFIH1 IFIT1 IFIT2 IFIT3 IFITM1 IFITM2 IFITM3 IFNA1 IFNA10 IFNA14
IFNA16 IFNA17 IFNA2 IFNA21 IFNA4 IFNA5 IFNA6 IFNA7 IFNA8 IFNAR2 IFNB1 IFNE IFNGR1 IFNGR2 IFNK IFNLR1 IFNW1
IKBKB IL10 IL10RA IL10RB IL11 IL11RA IL12A IL12B IL12RB1 IL12RB2 IL13 IL13RA1 IL13RA2 IL15 IL15RA IL16 IL17F
IL17RA IL17RC IL18 IL18BP IL18R1 IL18RAP IL1A IL1B IL1F10 IL1R1 IL1RAP IL1RL1 IL2 IL20RA IL20RB IL21R IL22RA1
IL22RA2 IL23R IL2RA IL2RB IL2RG IL3 IL31RA IL33 IL34 IL36A IL36B IL36G IL37 IL3RA IL4 IL4R IL5 IL5RA IL6 IL6R
IL6ST IL7 IL7R IL9 IL9R ILK INS IRAK1 IRAK2 IRAK3 IRAK4 IRF1 IRF2 IRF4 IRF5 IRF7 IRF8 IRF9 IRS1 ISG15 ISG20
ISOC1 ITGB7 ITK JAG2 JAGN1 JAK1 JAK2 JAK3 JAML KIF13B KIT KLRK1 LAMP3 LAP3 LATS2 LCK LCP2 LEP LEPROT LFNG
LGALS3BP LIFR LILRA1 LILRA2 LILRA3 LILRA4 LILRA5 LILRA6 LILRB1 LILRB2 LILRB4 LILRB5 LIMS1 LMBR1L LPAR6 LRP8
LY6E LY9 LYN LYSMD2 MAP3K7 MAPK3 MARCHF1 MDK METTL7B MICA MICB MOV10 MSN MT2A MT3 MTHFD2 MVB12A MVP MX1 MX2
MYD88 NAMPT NCK1 NCK2 NCOA3 NCOA7 NCOR2 NFKB1 NFKBIA NHEJ1 NLRC3 NLRC5 NMI NOD1 NUB1 NUMBL NUP93 OAS1 OAS2
OAS3 OASL OGFR OSMR P2RY14 P3R3URF P4HB PARP12 PARP14 PARP9 PDE4B PDGFB PELI1 PF4 PFKP PIK3CG PIK3R1 PIK3R2
PIK3R3 PIM1 PLA2G4A PLCB1 PLP2 PLSCR1 PLVAP PML PNP PNPT1 PPP3CA PPP3CB PRKCA PRL PRLR PROCR PSMA2 PSMA3
PSMB10 PSMB2 PSMB8 PSMB9 PSME1 PSME2 PTGS2 PTK2B PTPN1 PTPN11 PTPN2 PTPN22 PTPN6 PTPRC PTPRJ PYCARD RAB29
RAPGEF6 RASGRP1 RBCK1 RBX1 RELA RHEX RHOH RIPK1 RIPK2 RNF213 RNF31 RPS6KA4 RPS6KA5 RRAGA RSAD2 RTP4 SAMD9
SAMD9L SAMHD1 SCRIB SECTM1 SELL SELP SERPING1 SIRT1 SLA2 SLAMF6 SLAMF7 SLAMF9 SLC25A28 SMAD4 SOCS1 SOCS2 SOCS3
SOCS4 SOCS5 SOD2 SOS1 SOX4 SP110 SPI1 SPNS1 SPPL2A SRC SRI SSPN ST18 ST3GAL5 ST8SIA4 STAT1 STAT2 STAT3 STAT4
STAT5A STAT5B STAT6 SYK TANK TAP1 TAPBP TARP TDRD7 TENT5A TFF2 TMEM140 TMSB4X TNF TNFAIP2 TNFAIP3 TNFAIP6
TNFRSF11A TNFRSF1A TNFRSF1B TNFRSF4 TNFSF10 TNFSF11 TNFSF14 TNFSF18 TNFSF4 TNIP2 TOLLIP TOR1B TOX4 TP53 TRADD
TRAF1 TRAF3 TRAF3IP2 TRAF6 TRAFD1 TREML2 TRIM14 TRIM21 TRIM25 TRIM26 TRIM5 TSLP TXNDC17 TXNIP TYK2 UBA7 UBE2L6
UGCG UPP1 USP18 VAMP5 VAMP8 VCAM1 WARS1 XAF1 XCL1 XCL2 ZAP70 ZBP1 ZNFX1
""".split()

In [ ]:
# The list of immune genes (previous cell) ∩ the genes that are expressed in our data (Part 2 filter)
if adata.n_vars > 2000:                 # keep the complete object aside (re-running this cell is safe)
    adata_all = adata
adata = adata_all[:, adata_all.var_names.isin(IMMUNE_GENES)].copy()
print(f"Immune gene list: {len(set(IMMUNE_GENES))} genes  →  {adata.n_vars} are expressed in our data (of {adata_all.n_vars:,})")
for g in (EXAMPLE_GENE, CONTROL_GENE):
    assert g in adata.var_names, f"{g} is not in the immune gene list: choose another gene in the settings cell"
print(adata)

## 4.1 · The paired design, one gene at a time

Each donor was measured **with PBS and with every cytokine**. The two measurements of a donor are *not independent*: they share the donor's genetics, sex, age, baseline immune state…
(Part 3 showed strong donor effects.) A **paired analysis** compares every donor **with itself**, so these differences cancel out. With $y=\log_2(\text{CPM}+1)$:

$$d_{gi}\;=\;y^{\text{cytokine}}_{gi}-y^{\text{PBS}}_{gi}\qquad(g=\text{gene},\; i=\text{donor})$$

We now have **12 numbers per gene** (one per donor), and the question becomes: *is the mean of these 12 differences different from 0?*
Let's follow `EXAMPLE_GENE` in `CELL_TYPE`, from the data to the p-value.

In [ ]:
pbs_names = [f"{d}|{CELL_TYPE}|PBS" for d in DONORS]               # the 12 control samples (one per donor)
cyt_names = [f"{d}|{CELL_TYPE}|{CYTOKINE}" for d in DONORS]        # the 12 treated samples, in the SAME donor order

x_pbs = adata[pbs_names, EXAMPLE_GENE].X.ravel().astype(float)     # log2(CPM + 1) of the gene in each donor, with PBS…
x_cyt = adata[cyt_names, EXAMPLE_GENE].X.ravel().astype(float)     # …and with the cytokine
d     = x_cyt - x_pbs                                              # the paired difference of every donor

display(pd.DataFrame({"PBS": x_pbs, CYTOKINE: x_cyt, "difference d": d}, index=DONORS).round(2).T)

fig, axes = plt.subplots(1, 2, figsize=(8.5, 4.2))
paired_plot(axes[0], x_pbs, x_cyt, EXAMPLE_GENE, CYTOKINE)
ctrl_pbs = adata[pbs_names, CONTROL_GENE].X.ravel().astype(float); ctrl_cyt = adata[cyt_names, CONTROL_GENE].X.ravel().astype(float)
paired_plot(axes[1], ctrl_pbs, ctrl_cyt, CONTROL_GENE, CYTOKINE)
fig.suptitle(f"{CT_LABEL[CELL_TYPE]}: each line is one donor (red: higher with the cytokine, blue: lower)", y=1.02, fontsize=10)
plt.tight_layout(); plt.show()

**Left:** *every* donor goes up with the cytokine, even though the starting heights (the baselines) are very different from one donor to another: the paired analysis looks at the **change within each donor**.
**Right:** an immune gene that does not respond to IFN-γ (`IL10RA`): lines go both ways, with no consistent change.

### From 12 differences to a p-value

$$\log_2\text{FC}_g=\bar d_g=\frac1n\sum_i d_{gi},\qquad s_g=\sqrt{\frac{1}{n-1}\sum_i (d_{gi}-\bar d_g)^2},\qquad t_g=\frac{\bar d_g}{s_g/\sqrt n},\qquad p_g = 2\,\Pr\!\bigl(T_{n-1}>|t_g|\bigr)$$

Under the **null hypothesis** (the cytokine does nothing) $t_g$ follows a Student *t* distribution with $n-1=11$ degrees of freedom. The **p-value** is the probability of getting a $|t|$ at least as large as ours *if the null were true*.

In [ ]:
n      = len(d)                          # 12 donors
mean_d = d.mean()                        # = the log2 fold change
sd_d   = d.std(ddof=1)                   # spread of the differences
se     = sd_d / np.sqrt(n)               # standard error of the mean difference
t_obs  = mean_d / se
p_val  = 2 * stats.t.sf(abs(t_obs), df=n - 1)

print(f"{EXAMPLE_GENE} in {CT_LABEL[CELL_TYPE]}, {CYTOKINE} vs PBS")
print(f"  mean difference (log2FC) = {mean_d:.3f}   → fold change = {2 ** mean_d:.1f}×")
print(f"  SD of the differences    = {sd_d:.3f}")
print(f"  standard error           = {sd_d:.3f} / √{n} = {se:.3f}")
print(f"  t = {mean_d:.3f} / {se:.3f} = {t_obs:.2f}        (degrees of freedom = {n - 1})")
print(f"  p = 2 · Pr(T > {abs(t_obs):.2f}) = {p_val:.1e}")
ref = stats.ttest_rel(x_cyt, x_pbs)
print(f"\nscipy.stats.ttest_rel gives t = {ref.statistic:.2f}, p = {ref.pvalue:.1e}   (identical ✔)")

xx = np.linspace(-max(6, abs(t_obs) * 1.25), max(6, abs(t_obs) * 1.25), 500)
fig, ax = plt.subplots(figsize=(7, 3.3))
ax.plot(xx, stats.t.pdf(xx, df=n - 1), color="k", lw=1.5)
for sgn in (1, -1):
    tail = xx[sgn * xx >= abs(t_obs)]
    ax.fill_between(tail, stats.t.pdf(tail, df=n - 1), color=UP_COLOR, alpha=0.7)
ax.axvline(t_obs, color=UP_COLOR, ls="--")
ax.annotate(f"observed t = {t_obs:.1f}\np = {p_val:.1e}", xy=(t_obs, 0.2), xytext=(-8, 0), textcoords="offset points", ha="right", color=UP_COLOR, fontsize=9)
ax.set_xlabel("t statistic"); ax.set_ylabel("density"); ax.set_title(f"t distribution with {n - 1} degrees of freedom (null hypothesis: no effect)")
plt.tight_layout(); plt.show()

The shaded tails are the probability of seeing something *at least this extreme* by chance: tiny, so the data are hard to reconcile with "no effect".

#### 🔧 Exercise 4.1 · Your turn

> Change `EXAMPLE_GENE` (e.g. to `"IDO1"`, `"STAT1"` or `"IL10RA"`) in the *settings* cell, then re-run the three cells above. Which gets the **smallest p-value**: the gene with the largest fold change, or the gene whose differences are most **consistent** across donors?

## 4.2 · The same question for all genes: DESeq2

A t-test on log-CPM is a good teaching tool, but real studies use tools designed for **counts**. The most popular is **DESeq2** (R); `pydeseq2` is its Python implementation. At a high level it does four things:

| Step | What DESeq2 does | Why |
|:--:|---|---|
| **1. Size factors** | Works on the **raw counts** and rescales every sample with the *median-of-ratios* method: a sample's size factor is the median, across genes, of its ratio to the gene's average over all samples | A robust version of CPM: it assumes that *most genes do not change*, so a few strongly induced genes cannot distort the scale (the composition effect of Part 2) |
| **2. Model** | Fits for every gene a **negative binomial** generalized linear model: $\log \mu_{gi}=\log s_i+\beta_{g,\text{donor}(i)}+\beta_{g,\text{cytokine}(i)}$ | Counts are not normally distributed: variance grows with the mean. `~ donor + cytokine` is the **paired design**: each donor gets its own baseline, and `cytokine` is the effect we want |
| **3. Dispersion** | Estimates the noise of each gene and **shrinks it towards a trend** shared by all genes | With only 12 donors, a gene's own noise estimate is unreliable; borrowing information across genes makes the test more stable |
| **4. Test** | A **Wald test** of the cytokine coefficient: $\log_2\text{FC}$, its standard error, a p-value | Same logic as the t-test: *is the effect larger than the noise?* |

We then correct the p-values for the ~450 genes we test (next section). The function below runs everything for one cytokine against another condition in one cell type. It applies **the same expression filter** as Part 2 to the 24 samples of the comparison.

In [ ]:
from pydeseq2.dds import DeseqDataSet
from pydeseq2.ds import DeseqStats

def run_deseq2(adata, cell_type, treated, reference, padj_thresh=PADJ_THRESH, lfc_thresh=LFC_THRESH):
    '''DESeq2 (pydeseq2): `treated` vs `reference` in one cell type, 12 donors, design "~ donor + cytokine". One row per gene.'''
    names = [f"{d}|{cell_type}|{c}" for c in (reference, treated) for d in DONORS]         # 12 reference + 12 treated samples
    sub   = adata[names]
    counts = pd.DataFrame(sub.layers["counts"].astype(int), index=names, columns=adata.var_names)     # RAW counts, integers
    counts = counts.loc[:, is_expressed(counts.values)]                                    # expression filter (Part 2) on these 24 samples
    meta = pd.DataFrame({"donor": DONORS * 2, "cytokine": [reference] * 12 + [treated] * 12}, index=names)

    dds = DeseqDataSet(counts=counts, metadata=meta, design="~ donor + cytokine", quiet=True)
    dds.deseq2()                                                                           # size factors → dispersions → model fit
    stat = DeseqStats(dds, contrast=["cytokine", treated, reference], quiet=True)          # Wald test of "treated vs reference"
    stat.summary()                                                                         # p-values and Benjamini–Hochberg adjusted p-values

    res = stat.results_df.rename(columns={"log2FoldChange": "log2fc", "pvalue": "pval", "baseMean": "base_mean"}).reindex(adata.var_names)
    for col, name in [("genewise_dispersions", "disp_genewise"), ("fitted_dispersions", "disp_trend"), ("dispersions", "disp_final")]:
        res[name] = dds.var[col].reindex(adata.var_names)                                  # the three steps of the dispersion estimate
    res["tested"] = res["padj"].notna()
    res["direction"] = "ns"
    res.loc[res["tested"] & (res["padj"] < padj_thresh) & (res["log2fc"] >  lfc_thresh), "direction"] = "up"
    res.loc[res["tested"] & (res["padj"] < padj_thresh) & (res["log2fc"] < -lfc_thresh), "direction"] = "down"
    res.attrs["size_factors"] = dds.obs["size_factors"].to_dict()                          # kept to look at them below
    res.attrs["n_filtered_in"] = counts.shape[1]
    return res

de = run_deseq2(adata, CELL_TYPE, CYTOKINE, "PBS")          # ~ 3 seconds

print(f"{CYTOKINE} vs PBS in {CT_LABEL[CELL_TYPE]}")
print(f"  genes in the object        : {len(de):,}")
print(f"  pass the expression filter : {de.attrs['n_filtered_in']:,}")
print(f"  tested (with an adj. p)    : {de['tested'].sum():,}")
print(f"  up-regulated (DEGs)        : {(de['direction'] == 'up').sum():,}")
print(f"  down-regulated (DEGs)      : {(de['direction'] == 'down').sum():,}")
with pd.option_context("display.float_format", "{:.3g}".format):
    print("\nOur two example genes, in the results table:"); display(de.loc[[EXAMPLE_GENE, CONTROL_GENE]])
    print("The 8 genes with the smallest p-value:");        display(de[de["tested"]].sort_values("pval").head(8))

| Column | Meaning |
|---|---|
| `base_mean` | average of the *normalized* counts over the 24 samples |
| `log2fc` | $\log_2$ fold change, `treated` vs `reference` (+1 = doubles) |
| `lfcSE`, `stat`, `pval` | standard error, Wald statistic and its p-value: **how surprising** the change is if nothing happens |
| `padj` | p-value **adjusted for multiple testing** (Benjamini–Hochberg): see next section |
| `tested` | the gene has an adjusted p-value (it passed the filters) |
| `direction` | `up` / `down` if `padj < PADJ_THRESH` **and** `|log2fc| > LFC_THRESH`, otherwise `ns` (not significant) |

**How different is it from our log-CPM t-test?** Let's run both on all genes and compare:

In [ ]:
def paired_ttest_de(adata, cell_type, treated, reference):
    '''The paired t-test of 4.1 on log2(CPM + 1), for all genes at once (+ BH correction).'''
    ir = adata.obs_names.get_indexer([f"{d}|{cell_type}|{reference}" for d in DONORS])
    it = adata.obs_names.get_indexer([f"{d}|{cell_type}|{treated}" for d in DONORS])
    diff   = adata.X[it].astype(np.float64) - adata.X[ir]                                   # 12 donors × genes
    log2fc = diff.mean(axis=0)
    se     = diff.std(axis=0, ddof=1) / np.sqrt(len(DONORS))
    with np.errstate(divide="ignore", invalid="ignore"):
        t = log2fc / se
    pval   = np.where(se > 0, 2 * stats.t.sf(np.abs(t), df=len(DONORS) - 1), 1.0)
    tested = is_expressed(np.vstack([adata.layers["counts"][ir], adata.layers["counts"][it]]))
    padj   = np.full(len(pval), np.nan); padj[tested] = multipletests(pval[tested], method="fdr_bh")[1]
    return pd.DataFrame({"log2fc": log2fc, "pval": pval, "padj": padj, "tested": tested}, index=adata.var_names)

de_t = paired_ttest_de(adata, CELL_TYPE, CYTOKINE, "PBS")
both = de["tested"] & de_t["tested"]

sig_d = (de["padj"] < PADJ_THRESH) & (de["log2fc"].abs() > LFC_THRESH) & de["tested"]
sig_t = (de_t["padj"] < PADJ_THRESH) & (de_t["log2fc"].abs() > LFC_THRESH) & de_t["tested"]
print(f"DEGs: DESeq2 {sig_d.sum():,} · t-test {sig_t.sum():,} · both {(sig_d & sig_t).sum():,}")
print(f"Median log2FC of the tested genes: DESeq2 {de.loc[de['tested'], 'log2fc'].median():+.3f} · log-CPM t-test {de_t.loc[de_t['tested'], 'log2fc'].median():+.3f}")

fig, axes = plt.subplots(1, 3, figsize=(17, 4.3))
ax = axes[0]
ax.scatter(de_t.loc[both, "log2fc"], de.loc[both, "log2fc"], s=3, alpha=0.3, color="#2a78d6", linewidths=0, rasterized=True)
lim = 4.5; ax.plot([-lim, lim], [-lim, lim], color="k", lw=0.8, ls="--"); ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim)
r = np.corrcoef(de_t.loc[both, "log2fc"], de.loc[both, "log2fc"])[0, 1]
ax.set_xlabel("log2FC, log-CPM difference"); ax.set_ylabel("log2FC, DESeq2"); ax.set_title(f"Effect sizes (r = {r:.2f})")

# size factors against library size: DESeq2's normalization is a robust version of CPM
sf = pd.Series(de.attrs["size_factors"]); libsz = adata.obs.loc[sf.index, "lib_size"].values
ax = axes[1]
is_ref = np.array([n.endswith("|PBS") for n in sf.index])
ax.scatter(libsz[is_ref], sf.values[is_ref], s=40, color=PBS_COLOR, label="PBS"); ax.scatter(libsz[~is_ref], sf.values[~is_ref], s=40, color=CYTOKINE_COLOR, label=CYTOKINE)
ax.set_xscale("log"); ax.set_yscale("log"); ax.set_xlabel("library size"); ax.set_ylabel("DESeq2 size factor"); ax.legend(loc="upper left")
ax.set_title(f"Size factors follow the library size (r = {np.corrcoef(np.log(libsz), np.log(sf.values))[0, 1]:.3f})")

# dispersion: the noise of every gene. Raw estimate (grey) → shrunk towards the trend (red line) → final value (blue)
dd = de[de["tested"] & (de["disp_genewise"] > 1e-6)]
ax = axes[2]
ax.scatter(dd["base_mean"], dd["disp_genewise"], s=3, alpha=0.4, color=PBS_COLOR, linewidths=0, rasterized=True)
ax.scatter(dd["base_mean"], dd["disp_final"], s=3, alpha=0.25, color="#2a78d6", linewidths=0, rasterized=True)
o_ = np.argsort(dd["base_mean"].values); ax.plot(dd["base_mean"].values[o_], dd["disp_trend"].values[o_], color=UP_COLOR, lw=2)
ax.set_xscale("log"); ax.set_yscale("log"); ax.set_xlabel("mean of normalized counts"); ax.set_ylabel("dispersion (noise)"); ax.set_title("Dispersion: shrunk towards the trend")
ax.legend(handles=[Line2D([], [], marker="o", color=PBS_COLOR, ls="", label="gene-wise estimate"), Line2D([], [], marker="o", color="#2a78d6", ls="", label="final (shrunk)"),
                   Line2D([], [], color=UP_COLOR, lw=2, label="trend")], loc="lower left", fontsize=8)
plt.tight_layout(); plt.show()

#### ❓ Exercise 4.2 · Questions
1. The two approaches give similar effect sizes. Where do they differ most? Look at the **median log₂FC** printed above: which one is closer to 0?
2. Why are the size factors almost proportional to the library size (middle panel)? What is different about them?
3. Right panel: what happens to the noisy gene-wise dispersion estimates (grey) after DESeq2 shrinks them towards the trend? Why does that help with 12 donors?

<details>
<summary>💡 Answers</summary>

1. The sizes of the effects agree well, but the medians differ: DESeq2's is ≈ 0.00 whereas the log-CPM one is **+0.27**. In this immune-gene selection many genes go up with IFN-γ, so the *typical* gene looks higher in log-CPM; DESeq2 takes the median gene as the "no change" reference (median-of-ratios), so its median is 0 by construction. ⚠️ With a selection enriched in responders that assumption is weaker: size factors estimated on these genes only may hide part of the real up-regulation (the *composition effect* of Part 2, bonus B2). DESeq2 also gives more weight to genes with many counts and less to noisy low-count ones.
2. Because the library size *is* the main technical difference between samples; the size factors are a robust version of it, estimated from the genes that do not change instead of from the total.
3. The grey cloud is wide (with 12 samples each gene's own estimate is noisy); the blue points are pulled towards the red trend, so the extreme values are tamed. Fewer genes look "too noisy" or "too clean" by chance, and the p-values are more reliable.

</details>

## 4.3 · Multiple testing: why we correct the p-values

Each gene is tested at the 5 % level: *if nothing happened at all*, 5 % of the genes would still get $p<0.05$ by chance. With ~450 tested genes that is **~22 false positives** (with the whole transcriptome, ~12,700 genes, it would be ~640: another reason why the correction matters even more there). The **False Discovery Rate (FDR)** is the expected fraction of false positives *among the genes we call significant*.

In [ ]:
tested = de["tested"].values
n_tested = tested.sum()
print(f"Genes tested: {n_tested:,}   →   false positives expected at p < 0.05 if NOTHING happened: {0.05 * n_tested:,.0f}")
print(f"Genes with raw p < 0.05            : {(de.loc[tested, 'pval'] < 0.05).sum():,}")
print(f"Genes with adjusted p (padj) < 0.05: {(de.loc[tested, 'padj'] < 0.05).sum():,}")

fig, ax = plt.subplots(figsize=(6.5, 3.5))
ax.hist(de.loc[tested, "pval"], bins=40, range=(0, 1), color="#2a78d6")
ax.axhline(n_tested / 40, color="k", ls="--", lw=0.8); ax.annotate("what we would see if nothing happened", xy=(0.97, n_tested / 40), xytext=(0, 4), textcoords="offset points", ha="right", fontsize=8)
ax.set_xlabel("p-value"); ax.set_ylabel("number of genes"); ax.set_title(f"{CYTOKINE} vs PBS · {CT_LABEL[CELL_TYPE]}")
plt.tight_layout(); plt.show()

The flat floor is the "noise" (the dashed line); the **spike near 0** are the genes that truly respond.

**Benjamini–Hochberg (BH)** controls the FDR: among the genes we call significant at `padj < 0.05`, we *expect* at most 5 % to be false positives.
With $m$ tests and p-values sorted $p_{(1)}\le\dots\le p_{(m)}$:

$$p^{\text{adj}}_{(i)}\;=\;\min_{k\ge i}\Bigl(\tfrac{m}{k}\,p_{(k)}\Bigr)\quad(\text{capped at }1)$$

i.e. *each p-value is multiplied by $m$ and divided by its rank* (then made monotone). A tiny example with 10 genes:

In [ ]:
p_toy = np.array([0.001, 0.008, 0.012, 0.030, 0.040, 0.045, 0.200, 0.400, 0.700, 0.900])      # 10 p-values, sorted
m, rank = len(p_toy), np.arange(1, len(p_toy) + 1)
step1 = p_toy * m / rank                                           # p × m / rank
bh    = np.minimum(np.minimum.accumulate(step1[::-1])[::-1], 1)    # make it monotone, from the largest rank down
display(pd.DataFrame({"p-value": p_toy, "rank": rank, "p × m / rank": step1.round(3), "BH adjusted": bh.round(3),
                      "statsmodels": multipletests(p_toy, method="fdr_bh")[1].round(3)}, index=[f"gene {i}" for i in rank]))
print(f"Raw p < 0.05: {(p_toy < 0.05).sum()} genes   →   adjusted p < 0.05: {(bh < 0.05).sum()} genes")

Six genes pass the raw 0.05 cutoff, but only three survive after correction: the others were *not strong enough given how many genes we tested*.

## 4.4 · What do we call "differentially expressed"? The volcano plot

A small p-value says that the effect is **unlikely to be zero**, not that it is **large**. With enough donors even a 1 % change gets a tiny p-value, but may be biologically irrelevant. So we ask for **both**:

$$\text{DEG}\;\Longleftrightarrow\;\texttt{padj}<\texttt{PADJ\_THRESH}\;\;\text{and}\;\;|\log_2\text{FC}|>\texttt{LFC\_THRESH}$$

The **volcano plot** shows both at once: *x* = log₂FC (effect size), *y* = −log₁₀ adjusted p-value (evidence).

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 6))
volcano(ax, de, f"{CYTOKINE} vs PBS · {CT_LABEL[CELL_TYPE]}")
plt.tight_layout(); plt.show()

#### ❓ Exercise 4.4 · Questions
1. Which genes did you recognise among the labels (the strongest changes)? Now look up `CXCL9`, `CXCL10`, `IDO1` and `STAT1` in the table `de`: are they DEGs? What is the biological role of these genes?
2. Genes in the **top centre** vs genes at the **sides but low**: what does each mean?
3. How many genes would be called DEG with `LFC_THRESH = 0` — and with `1`? (change it in the settings cell and re-run from 4.2). What does the fold-change threshold buy us?

<details>
<summary>💡 Answers</summary>

1. The labels are the DEGs with the best combination of evidence and effect size. In monocytes with IFN-γ, among the up-regulated ones you should find `CXCL9`, `LAMP3`, `EBI3`, `IL2RA` and `CCL5`, and among the down-regulated ones `CCL24`, `CCL13`, `CCL18`, `CSF1R` and `CD1C`: IFN-γ pushes the cells away from their resting monocyte state. `de.loc[["CXCL9", "CXCL10", "IDO1", "STAT1"]]` shows that the T-cell-attracting chemokines `CXCL9` and `CXCL10` and the tryptophan-degrading enzyme `IDO1` are DEGs (up ×4 to ×11): the classic **interferon-γ response**, i.e. recruitment of T cells and antimicrobial programmes. `STAT1` is *not* a DEG here (log₂FC ≈ +0.14, padj ≈ 0.34), although it is the transcription factor of that response.
2. **Top centre**: very significant but a *tiny* effect (small noise, small difference). **Sides but low**: big average changes that are *not consistent across donors* (not significant).
3. `LFC_THRESH = 0` keeps many more genes (about 310 here: all significant ones, including very small changes), `1` keeps only the strongest (about 70). The fold-change threshold removes statistically significant but biologically negligible changes.

</details>

## 4.5 · Comparing two cytokines

Everything so far compared a cytokine with PBS. Now we compare **two cytokines**: which genes do they *share*, and which genes are **specific** to each? For every cell type in `CELL_TYPES_COMPARE` we run DESeq2 three times (the plots and the table only use the genes that were tested in *both* comparisons):

* `CYTOKINE_A` vs PBS and `CYTOKINE_B` vs PBS, then we put the two fold changes side by side;
* `CYTOKINE_A` vs `CYTOKINE_B` **directly**: the genes that differ between the two cytokines.

In [ ]:
# 🔧 CHANGE ME (in the settings cell above): CYTOKINE_A, CYTOKINE_B and CELL_TYPES_COMPARE (1 or 2 cell types). ~ 7 seconds per cell type
cmp = {}
for ct in CELL_TYPES_COMPARE:
    cmp[ct] = dict(A=run_deseq2(adata, ct, CYTOKINE_A, "PBS"), B=run_deseq2(adata, ct, CYTOKINE_B, "PBS"), AB=run_deseq2(adata, ct, CYTOKINE_A, CYTOKINE_B))

COL_A, COL_B, COL_BOTH = "#2a78d6", "#eb6834", "#6250d6"
fig, axes = plt.subplots(len(CELL_TYPES_COMPARE), 2, figsize=(12, 5.2 * len(CELL_TYPES_COMPARE)), squeeze=False)
rows = []
for r_, ct in enumerate(CELL_TYPES_COMPARE):
    A, B, AB = cmp[ct]["A"], cmp[ct]["B"], cmp[ct]["AB"]
    ok = A["tested"] & B["tested"]                                      # only genes tested in both comparisons are compared
    sig_a, sig_b = (A["direction"] != "ns") & ok, (B["direction"] != "ns") & ok
    ax = axes[r_, 0]
    for mask_, col, lab in [(ok & ~sig_a & ~sig_b, NS_COLOR, "neither"), (ok & sig_a & ~sig_b, COL_A, f"{CYTOKINE_A} only"),
                            (ok & ~sig_a & sig_b, COL_B, f"{CYTOKINE_B} only"), (ok & sig_a & sig_b, COL_BOTH, "both")]:
        ax.scatter(A.loc[mask_, "log2fc"], B.loc[mask_, "log2fc"], s=5, alpha=0.55, color=col, linewidths=0, label=f"{lab} ({mask_.sum():,})", rasterized=True)
    ax.axhline(0, color="k", lw=0.6); ax.axvline(0, color="k", lw=0.6)
    ax.set_xlabel(f"log2FC  {CYTOKINE_A} vs PBS"); ax.set_ylabel(f"log2FC  {CYTOKINE_B} vs PBS"); ax.set_title(f"{CT_LABEL[ct]}: shared and specific responses")
    ax.legend(loc="upper left", markerscale=3, fontsize=8)
    volcano(axes[r_, 1], AB, f"{CT_LABEL[ct]}: {CYTOKINE_A} vs {CYTOKINE_B}", n_label=10, xlabel=f"log2 fold change ({CYTOKINE_A} / {CYTOKINE_B})")
    rows.append({"cell type": CT_LABEL[ct], f"DEGs {CYTOKINE_A}": int(sig_a.sum()), f"DEGs {CYTOKINE_B}": int(sig_b.sum()),
                 "shared": int((sig_a & sig_b).sum()), f"{CYTOKINE_A} only": int((sig_a & ~sig_b).sum()), f"{CYTOKINE_B} only": int((~sig_a & sig_b).sum()),
                 f"DEGs {CYTOKINE_A} vs {CYTOKINE_B}": int((AB["direction"] != "ns").sum())})
plt.tight_layout(); plt.show()
display(pd.DataFrame(rows).set_index("cell type"))

#### ❓ Exercise 4.5 · Questions
1. Look at the left plots. Genes along the diagonal respond *the same way* to both cytokines; genes near an axis respond to only one (colours: blue = only A, orange = only B, violet = both). What do you see for IFN-γ and IL-4? Are all the "both" genes on the diagonal?
2. Do the two cytokines act more differently in monocytes or in CD4 T cells? How can you tell from the table?
3. The **right-hand volcano** compares the two cytokines directly. Which genes are higher with IFN-γ than with IL-4? Do they match the *IFN-γ only* genes on the left?

<details>
<summary>💡 Answers</summary>

1. The cloud is **not** concentrated on the diagonal: in monocytes, of the ~190 DEGs of IFN-γ and ~250 of IL-4, only ~120 are shared; the rest are specific to one cytokine (see the legend and the table). Even among the "both" genes (violet) most (~90 of ~120) go in **opposite directions** (lower-right quadrant: up with IFN-γ, down with IL-4), which fits the two programmes being different (Part 2, heatmap of immune genes).
2. In **monocytes**: they have more DEGs with each cytokine (~190 and ~250, against ~135 and ~135 in CD4 T cells) and also more genes that differ between the two cytokines (last column: ~330 against ~220). T cells respond less to both.
3. Check the labels of the volcano: in monocytes the strongest genes on the right (higher with IFN-γ) include `IDO1`, `FCGR1A`, `CXCL9` and `CXCL11` (classic IFN-γ targets), and on the left (higher with IL-4) the chemokines `CCL24`, `CCL23`, `CCL18` and `CCL26`, typical of IL-4-driven ("alternative") macrophage activation, and `CD1C`. Many of these are among the "only A" and "only B" genes of the left plot, because a gene that responds to one cytokine and not to the other differs between them.

</details>

## 4.6 · 🔧 Your turn

1. In the **settings** cell, change `CYTOKINE` to `"IL-4"` and `EXAMPLE_GENE` to `"CCL26"`, and re-run from 4.1. Which of your conclusions still hold?
2. Pick **any two cytokines** of the 87 (`CYTOKINE_A`, `CYTOKINE_B`), for example two interferons (`"IFN-alpha1"` and `"IFN-gamma"`) or two cytokines of the same family, and one or two cell types. Which pair is the most similar? The most different?
3. Change `PADJ_THRESH` to `0.01` and `LFC_THRESH` to `1`. How does the number of DEGs change?

## 4.7 · Take-home messages

| | |
|---|---|
| **Design** | Pseudobulk: the sum of the cells of a donor. The unit of replication is the **donor**, not the cell. `~ donor + cytokine` is a paired design: it removes the shared donor effects. |
| **Tool** | **DESeq2** works on raw counts (median-of-ratios size factors, negative binomial model, shrunken dispersions) and is the standard for real studies. `limma-voom` and `edgeR` are alternatives. |
| **Multiple testing** | ~450 tests here (~12,700 with all genes): always report **adjusted** p-values (FDR, Benjamini–Hochberg). |
| **Significance ≠ importance** | Combine `padj` with an effect-size threshold, and look at individual donors. |
| **Filtering** | Remove untestable genes *before* the correction. Restricting to a gene set (here, immune genes) lightens the correction, but it must be decided **before** looking at the results and it biases the fraction of DEGs. |
| **Next step** | A list of DEGs becomes biology through **functional enrichment** (pathways, transcription-factor targets). |

---
# 🏁 Wrap-up

You went from a single number to a differential-expression result:

**raw counts** → `AnnData` → **library sizes** → **CPM and log₂** → **gene filter** → **PCA** (what varies?) → **DESeq2 + FDR** (what changes?) → **two cytokines** (what is shared, what is specific?)

**Where to go from here**

* **Functional enrichment** of the DEG lists (pathways, transcription-factor targets): `gseapy`, `decoupler`.
* **Single-cell analysis** of the original data: `scanpy`; best practices in Heumos *et al.* (2023), *Nat Rev Genet*.
* **The Human Cytokine Dictionary** itself and its `huCIRA` package (Oesinghaus *et al.*, 2025) to interpret your own data with cytokine-response signatures.

**References**
* Oesinghaus L. *et al.* (2025) A single-cell cytokine dictionary of human peripheral blood. *bioRxiv*, doi:10.64898/2025.12.12.693897
* Virshup I. *et al.* (2024) anndata: Access and store annotated data matrices. *J. Open Source Softw.* 9:4371
* Heumos L. *et al.* (2023) Best practices for single-cell analysis across modalities. *Nat. Rev. Genet.* 24:550–572
* Love M.I., Huber W., Anders S. (2014) Moderated estimation of fold change and dispersion for RNA-seq data with DESeq2. *Genome Biol.* 15:550
* Muzellec B. *et al.* (2023) PyDESeq2: a python package for bulk RNA-seq differential expression analysis. *Bioinformatics* 39:btad547
* Benjamini Y., Hochberg Y. (1995) Controlling the false discovery rate. *J. R. Stat. Soc. B* 57:289–300

In [ ]:
# Back to ALL the expressed genes for the bonus sections (Part 4 only used the immune genes)
adata = adata_all
print(adata)

---
# 🎁 Bonus: only if you have time (or at home)

## B1 · The PBS puzzle: always check how your control was built

In 2.1 we dropped 6 of the 7 PBS rows of every donor and cell type. *Why was that right?* Let's look at the raw file.

In [ ]:
import scipy.sparse as sp

raw_obs = adata_raw.obs
ex = raw_obs[(raw_obs["cytokine"] == "PBS") & (raw_obs["donor"] == "Donor1") & (raw_obs["cell_type"] == "CD14_Mono")]
print("The 7 PBS rows of Donor1 / CD14_Mono:")
display(ex[["sample", "bc1_well", "psbulk_n_cells"]])
print(f"6 individual wells (H7–H12) → {ex.loc[ex['bc1_well'].notna(), 'psbulk_n_cells'].sum():,.0f} cells in total;   the row with no well has {ex.loc[ex['bc1_well'].isna(), 'psbulk_n_cells'].iloc[0]:,.0f} cells")

The 7th row looks like the **sum of the six wells**. Let's *verify* it in all donor × cell-type groups instead of trusting it:

In [ ]:
is_pbs, is_well = (raw_obs["cytokine"] == "PBS").values, raw_obs["bc1_well"].notna().values
Xr  = sp.csr_matrix(adata_raw.X)
pos = pd.Series(np.arange(len(raw_obs)), index=raw_obs.index)

n_groups, all_equal = 0, True
for (ct, donor), grp in raw_obs[is_pbs & is_well].groupby(["cell_type", "donor"], observed=True):
    sum_of_wells = np.asarray(Xr[pos[grp.index].values].sum(axis=0)).ravel()
    total_name   = raw_obs[is_pbs & ~is_well & (raw_obs["cell_type"] == ct) & (raw_obs["donor"] == donor)].index[0]
    all_equal   &= np.allclose(sum_of_wells, Xr[pos[total_name]].toarray().ravel())
    n_groups    += 1
print(f"Checked {n_groups} groups → the sum of the 6 wells equals the pre-computed PBS row in all of them: {all_equal}")

✅ So we keep the total row and drop the wells: **one PBS profile per donor and cell type**, like every cytokine.

> ⚠️ **Common trap:** if you summed *all seven* rows you would count every PBS cell twice. After CPM the result would happen to look the same, but the cell numbers would be wrong — exactly the kind of silent mistake that makes analyses irreproducible. *Check, don't assume.*

## B2 · What CPM cannot fix: composition effects

CPM is a **proportion**: if one gene takes a bigger share, *all the others* must take a smaller one. Both samples below have the same depth (10,000 counts) and **only gene E is truly induced** (6×):

In [ ]:
true = pd.DataFrame({"control": [2000, 2000, 2000, 2000, 2000], "treated": [2000, 2000, 2000, 2000, 12000]}, index=list("ABCDE"))
seq  = (true / true.sum() * 10_000).round()                      # what we observe at a fixed sequencing depth of 10,000
seq["log2FC observed"] = np.log2(seq["treated"] / seq["control"]).round(2)
seq["truth"] = ["unchanged"] * 4 + ["↑ 6×"]
display(seq)

Genes A–D did **not** change, yet they look 2× down because gene E now takes a large share of the sample: a **composition effect**. It is the reason for methods like *TMM* and *DESeq2's median-of-ratios*:
they assume that **most genes do not change** and rescale accordingly. Compare the distribution of the log₂FC of all genes with the two normalizations (the comparison of 4.2):

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.6))
bins = np.linspace(-3, 3, 90)
for res_, col, lab in [(de_t, PBS_COLOR, "log-CPM difference"), (de, "#2a78d6", "DESeq2")]:
    v = res_.loc[res_["tested"], "log2fc"]
    ax.hist(v, bins=bins, histtype="stepfilled", alpha=0.55, color=col, label=f"{lab} (median {v.median():+.2f})")
ax.axvline(0, color="k", lw=0.8); ax.set_xlabel("log2 fold change"); ax.set_ylabel("number of genes"); ax.legend(loc="upper left")
ax.set_title(f"{CYTOKINE} vs PBS · {CT_LABEL[CELL_TYPE]}")
plt.tight_layout(); plt.show()

With CPM the whole distribution is shifted slightly below 0 (a few very strongly induced genes take part of the library); DESeq2's size factors are centred. For a real study, use TMM / median-of-ratios, as DESeq2 does.

## B3 · A null experiment: how many false positives do we really get?

We make the data "cytokine-free": **(i)** remove the average effect of every gene and **(ii)** randomly flip the sign of each donor's remaining differences (if the cytokine did nothing, "cytokine − PBS" and "PBS − cytokine" would be equally likely).
The noise is the real noise, the effect is zero, and we run exactly the same paired t-test:

In [ ]:
def paired_pvalues(diff):
    n  = diff.shape[0]
    se = diff.std(axis=0, ddof=1) / np.sqrt(n)
    with np.errstate(divide="ignore", invalid="ignore"):
        t = diff.mean(axis=0) / se
    return np.where(se > 0, 2 * stats.t.sf(np.abs(t), df=n - 1), 1.0)

ip = adata.obs_names.get_indexer(pbs_names); ic = adata.obs_names.get_indexer(cyt_names)
diff  = (adata.X[ic] - adata.X[ip]).astype(np.float64)
noise = diff - diff.mean(axis=0)                                                    # (i) same noise, no effect
flip  = np.random.default_rng(1).choice([-1.0, 1.0], size=(len(DONORS), 1))         # (ii) one random sign per donor
tst   = de_t["tested"].values
p_null = paired_pvalues(noise * flip)[tst]
p_real = de_t.loc[tst, "pval"].values

fig, axes = plt.subplots(1, 2, figsize=(11, 3.6), sharey=True)
for ax, p, title, col in [(axes[0], p_real, "REAL comparison (t-test)", "#2a78d6"), (axes[1], p_null, "NULL: no effect, same noise", PBS_COLOR)]:
    ax.hist(p, bins=40, range=(0, 1), color=col); ax.axhline(len(p) / 40, color="k", ls="--", lw=0.8)
    ax.set_xlabel("p-value"); ax.set_title(f"{title}\n{(p < 0.05).sum():,} genes with p < 0.05")
axes[0].set_ylabel("number of genes")
plt.tight_layout(); plt.show()

padj_null = multipletests(p_null, method="fdr_bh")[1]
print(f"NULL data: {(p_null < 0.05).sum():,} genes with raw p < 0.05 (5 % of {len(p_null):,} = {0.05 * len(p_null):.0f} expected)")
print(f"NULL data: {(padj_null < 0.05).sum():,} genes with adjusted p < 0.05   ← after Benjamini–Hochberg")

On the null data **~5 % of the genes** have $p<0.05$ by pure chance, and the p-values are uniform (a flat histogram). After the correction, essentially **none** is called significant. That is what FDR control buys.

## B4 · Does pairing really help?

Pairing removes the donor's baseline **only if the baseline is shared by the two conditions**, i.e. if the PBS and cytokine values are **correlated across donors**. Compare the paired test with an **unpaired** (Welch) test, which ignores which samples come from the same donor:

In [ ]:
rows = []
for g in dict.fromkeys([EXAMPLE_GENE, "ISG15", "IFI6", "MX1", "STAT1", CONTROL_GENE]):
    a0 = adata[pbs_names, g].X.ravel().astype(float); a1 = adata[cyt_names, g].X.ravel().astype(float)
    rows.append({"gene": g, "log2FC": (a1 - a0).mean(), "r (PBS vs cytokine, across donors)": np.corrcoef(a0, a1)[0, 1],
                 "p paired": stats.ttest_rel(a1, a0).pvalue, "p unpaired": stats.ttest_ind(a1, a0, equal_var=False).pvalue})
with pd.option_context("display.float_format", "{:.3g}".format):
    display(pd.DataFrame(rows).set_index("gene"))

For the **interferon-stimulated genes** (`ISG15`, `IFI6`, `MX1`) the donors' baselines are strongly shared: some donors have high ISG expression *with and without* the cytokine.
The unpaired test drowns the effect in the between-donor spread, and the **paired test is much more sensitive**. Where PBS and cytokine values are *not* positively correlated, pairing brings nothing (and costs degrees of freedom).
Either way the experiment **is** paired, so the paired design (`~ donor + cytokine` in DESeq2) is the one that matches it.

## B5 · Free exploration

Pick **any cytokine** of the 87 and compare the four cell types (about 12 seconds).

In [ ]:
# 🔧 CHANGE ME: any cytokine of adata.obs["cytokine"]
EXPLORE_CYTOKINE = "IL-15"

if EXPLORE_CYTOKINE not in adata.obs["cytokine"].cat.categories:
    print(f"⚠️  '{EXPLORE_CYTOKINE}' not found. The cytokines are:\n", list(adata.obs["cytokine"].cat.categories))
else:
    de_explore = {ct: run_deseq2(adata, ct, EXPLORE_CYTOKINE, "PBS") for ct in CELL_TYPES}
    fig, axes = plt.subplots(2, 2, figsize=(12, 10), sharex=True, sharey=True)
    for ax, ct in zip(axes.ravel(), CELL_TYPES):
        volcano(ax, de_explore[ct], CT_LABEL[ct], n_label=10)
    fig.suptitle(f"{EXPLORE_CYTOKINE} vs PBS in the four cell types", fontweight="bold", y=0.995)
    plt.tight_layout(); plt.show()